# Multi-Feeder AC OPF in JuMP Using FeederFlow Data

This notebook builds AC OPF models directly from `FeederFlow.parse_file` outputs and solves IEEE 13/37/123 feeders.

Included constraints:
- Nodal AC power-balance equations
- Voltage magnitude bounds
- Branch current magnitude limits
- Branch angle-difference limits
- DER active/reactive/capability constraints
- Substation active/reactive/apparent power constraints

In [3]:
import Pkg

Pkg.activate(joinpath(@__DIR__, ".."))

required = ["JuMP", "Ipopt", "FeederFlow"]
project_deps = Set(String.(keys(Pkg.project().dependencies)))
for pkg in required
    if !(pkg in project_deps)
        Pkg.add(pkg)
    end
end

using FeederFlow
using JuMP
using Ipopt
using LinearAlgebra
using SparseArrays
using Printf

println("Environment ready")

  Activating project at `c:\Users\hoang\OneDrive - Massachusetts Institute of Technology\1. MIT\2. Projects\4. Dist OPF\multiphase_modelling\FeederFlow.jl`


Environment ready


## Model Construction Helpers

Implementation notes:
- Feeder parameters come from `NetworkModel` and `YBusModel` only.
- OPF state uses rectangular voltages on all nodes, with slack nodes fixed.
- Delta loads are mapped to an equivalent per-phase constant-power split on their connected phases for this notebook workflow.

In [71]:
phase_shift_deg(phase::Int) = phase == 1 ? 0.0 : (phase == 2 ? -120.0 : 120.0)

function build_balanced_slack_phasors(source::FeederFlow.SourceSpec)
    out = Dict{Int,ComplexF64}()
    for ph in source.phases
        ang = deg2rad(source.angle_deg + phase_shift_deg(ph))
        out[ph] = source.pu * cis(ang)
    end
    return out
end

function build_sparse_row_terms(Y::SparseMatrixCSC{ComplexF64,Int})
    n = size(Y, 1)
    rows, cols, vals = findnz(Y)
    terms = [Vector{Tuple{Int,ComplexF64}}() for _ in 1:n]
    for k in eachindex(rows)
        push!(terms[rows[k]], (cols[k], vals[k]))
    end
    return terms
end

function allocate_loads_to_nodes(network::FeederFlow.NetworkModel, ybus::FeederFlow.YBusModel)
    nall = length(ybus.all_order)
    p_load = zeros(Float64, nall)
    q_load = zeros(Float64, nall)

    for load in network.loads
        load.model == 2 && continue
        phases = load.phases
        isempty(phases) && continue
        share = 1.0 / length(phases)
        for ph in phases
            bp = FeederFlow.BusPhase(load.bus.bus, ph)
            idx = get(ybus.all_index, bp, 0)
            idx == 0 && continue
            p_load[idx] += share * load.p_pu
            q_load[idx] += share * load.q_pu
        end
    end

    return p_load, q_load
end

function build_generator_specs(network::FeederFlow.NetworkModel, ybus::FeederFlow.YBusModel)
    nall = length(ybus.all_order)
    node_map = [Vector{Tuple{Int,Float64}}() for _ in 1:nall]
    specs = NamedTuple[]

    for gen in network.generators
        phases = [ph for ph in gen.phases if haskey(ybus.all_index, FeederFlow.BusPhase(gen.bus.bus, ph))]
        isempty(phases) && continue

        pmax = max(gen.p_pu, 0.0)
        smax = max(gen.kva_pu, pmax + 1e-6)
        qcap = sqrt(max(smax^2 - pmax^2, 0.0))
        qmin = max(gen.qmin_pu, -qcap)
        qmax = min(gen.qmax_pu, qcap)
        if qmin > qmax
            qmin, qmax = -qcap, qcap
        end

        cost = max(gen.cost_coeff, 0.1)
        push!(specs, (
            name = gen.name,
            bus = gen.bus.bus,
            phases = phases,
            pmax = pmax,
            qmin = qmin,
            qmax = qmax,
            smax = smax,
            cost = cost,
        ))

        gid = length(specs)
        share = 1.0 / length(phases)
        for ph in phases
            idx = ybus.all_index[FeederFlow.BusPhase(gen.bus.bus, ph)]
            push!(node_map[idx], (gid, share))
        end
    end

    return specs, node_map
end

function estimate_substation_smax(network::FeederFlow.NetworkModel)
    kva_candidates = Float64[]
    for tr in vcat(collect(network.transformers), collect(network.regulators))
        for w in tr.windings
            if w.bus.bus == network.slack_bus
                push!(kva_candidates, w.kva)
            end
        end
    end

    raw = isempty(kva_candidates) ? 0.0 : maximum(kva_candidates) * 1000.0 / network.base.Sbase
    p_total = sum(l.p_pu for l in network.loads)
    q_total = sum(l.q_pu for l in network.loads)
    demand = hypot(p_total, q_total)
    return max(raw, 5.0 * demand + 0.2, 2.5)
end

function sanitize_dss_file_continuations!(file_path::AbstractString)
    lines = readlines(file_path)
    out = String[]
    buffer = ""
    continuing = false

    for raw in lines
        line = rstrip(raw)
        if continuing
            seg = strip(line)
            if endswith(seg, "\\")
                seg = rstrip(chop(seg; tail = 1))
                buffer *= " " * seg
            else
                buffer *= " " * seg
                push!(out, buffer)
                buffer = ""
                continuing = false
            end
        else
            if endswith(line, "\\")
                buffer = rstrip(chop(line; tail = 1))
                continuing = true
            else
                push!(out, raw)
            end
        end
    end

    continuing && push!(out, buffer)
    write(file_path, join(out, "\n") * "\n")
    return nothing
end

function copy_tree(src::AbstractString, dst::AbstractString)
    mkpath(dst)
    for (root, dirs, files) in walkdir(src)
        rel = relpath(root, src)
        dst_root = rel == "." ? dst : joinpath(dst, rel)
        mkpath(dst_root)
        for d in dirs
            mkpath(joinpath(dst_root, d))
        end
        for f in files
            cp(joinpath(root, f), joinpath(dst_root, f); force = true)
        end
    end
    return dst
end

function sanitized_dss_entry_path(dss_path::AbstractString)
    feeder_dir = dirname(dss_path)
    tmp_root = mktempdir()
    copied_dir = joinpath(tmp_root, basename(feeder_dir))
    copy_tree(feeder_dir, copied_dir)

    for (root, _, files) in walkdir(copied_dir)
        for f in files
            if endswith(lowercase(f), ".dss")
                sanitize_dss_file_continuations!(joinpath(root, f))
            end
        end
    end

    return joinpath(copied_dir, basename(dss_path))
end

function parse_network_with_fallback(dss_path::AbstractString)
    try
        return FeederFlow.parse_file(dss_path)
    catch err
        if err isa FeederFlow.DSSParseError
            return FeederFlow.parse_file(sanitized_dss_entry_path(dss_path))
        end
        rethrow()
    end
end

function build_and_solve_feeder_opf(dss_path::AbstractString; delta_deg::Float64 = 60.0, current_limit_scale::Float64 = 20.0)
    network = parse_network_with_fallback(dss_path)
    ybus = FeederFlow.build_y(network)

    nall = length(ybus.all_order)
    nnet = length(ybus.network_order)

    noload = FeederFlow.compute_no_load(ybus; v_slack = FeederFlow.source_slack(network.source, network.base))
    load_model = FeederFlow.build_load_model(network, ybus, noload)
    Yeff = copy(ybus.Ynet)
    Yeff[1:nnet, 1:nnet] .+= load_model.YL
    row_terms = build_sparse_row_terms(Yeff)

    slack_phasors = build_balanced_slack_phasors(network.source)

    p_load, q_load = allocate_loads_to_nodes(network, ybus)
    gen_specs, node_gen_map = build_generator_specs(network, ybus)
    ngen = length(gen_specs)

    vmin = fill(0.90, nall)
    vmax = fill(1.10, nall)
    for i in 1:nnet
        bp = ybus.all_order[i]
        vscale = FeederFlow.bus_vbase(network.base, bp.bus) / network.base.Vbase
        if haskey(network.buses, bp.bus)
            bus = network.buses[bp.bus]
            vmin[i] = bus.vmin_pu * vscale
            vmax[i] = bus.vmax_pu * vscale
        else
            vmin[i] *= vscale
            vmax[i] *= vscale
        end
    end

    function start_map_is_usable(map::Dict{FeederFlow.BusPhase,ComplexF64})
        isempty(map) && return false
        for (bp, v) in map
            local_mag = abs(v) * (network.base.Vbase / FeederFlow.bus_vbase(network.base, bp.bus))
            if !(isfinite(real(v)) && isfinite(imag(v)) && isfinite(local_mag) && local_mag <= 2.5)
                return false
            end
        end
        return true
    end

    start_map = Dict{FeederFlow.BusPhase,ComplexF64}()
    if isempty(start_map)
        try
            pf_bundle_loads = FeederFlow.solve_power_flow(build_network_without_generators(network); method = :zbus, max_iter = 80, tol = 1e-8)
            candidate = pf_bundle_loads.result.phase_voltages
            if start_map_is_usable(candidate)
                start_map = candidate
            end
        catch
        end
    end
    if isempty(start_map)
        try
            pf_bundle = FeederFlow.solve_power_flow(network; method = :zbus, max_iter = 80, tol = 1e-8)
            candidate = pf_bundle.result.phase_voltages
            if start_map_is_usable(candidate)
                start_map = candidate
            end
        catch
        end
    end

    start_vr = zeros(Float64, nall)
    start_vi = zeros(Float64, nall)
    for i in 1:nall
        bp = ybus.all_order[i]
        v_start = get(start_map, bp, get(slack_phasors, bp.phase, network.source.pu + 0im))
        start_vr[i] = real(v_start)
        start_vi[i] = imag(v_start)
    end

    model = JuMP.Model(Ipopt.Optimizer)
    set_silent(model)
    set_optimizer_attribute(model, "tol", 1e-8)
    set_optimizer_attribute(model, "max_iter", 6000)

    vbox = zeros(Float64, nall)
    for i in 1:nall
        bp = ybus.all_order[i]
        vbox[i] = 2.5 * FeederFlow.bus_vbase(network.base, bp.bus) / network.base.Vbase
    end

    @variable(model, vr[i=1:nall], lower_bound = -vbox[i], upper_bound = vbox[i])
    @variable(model, vi[i=1:nall], lower_bound = -vbox[i], upper_bound = vbox[i])

    for i in 1:nall
        set_start_value(vr[i], start_vr[i])
        set_start_value(vi[i], start_vi[i])
    end

    for bp in ybus.slack_order
        idx = ybus.all_index[bp]
        vref = get(slack_phasors, bp.phase, network.source.pu + 0im)
        @constraint(model, vr[idx] == real(vref))
        @constraint(model, vi[idx] == imag(vref))
    end

    for i in 1:nnet
        @constraint(model, vmin[i]^2 <= vr[i]^2 + vi[i]^2)
        @constraint(model, vr[i]^2 + vi[i]^2 <= vmax[i]^2)
    end

    Pg = JuMP.VariableRef[]
    Qg = JuMP.VariableRef[]
    if ngen > 0
        @variable(model, 0.0 <= Pg_var[g=1:ngen] <= gen_specs[g].pmax)
        @variable(model, gen_specs[g].qmin <= Qg_var[g=1:ngen] <= gen_specs[g].qmax)
        Pg = collect(Pg_var)
        Qg = collect(Qg_var)
        for g in 1:ngen
            @constraint(model, Pg[g]^2 + Qg[g]^2 <= gen_specs[g].smax^2)
            set_start_value(Pg[g], 0.0)
            set_start_value(Qg[g], 0.0)
        end
    end

    s_sub_max = estimate_substation_smax(network)
    @variable(model, -s_sub_max <= Pg_sub <= s_sub_max)
    @variable(model, -s_sub_max <= Qg_sub <= s_sub_max)
    @constraint(model, Pg_sub^2 + Qg_sub^2 <= s_sub_max^2)
    set_start_value(Pg_sub, min(s_sub_max, max(sum(p_load), 1e-3)))
    set_start_value(Qg_sub, 0.0)

    for i in 1:nnet
        ir = @expression(model, sum(real(y) * vr[j] - imag(y) * vi[j] for (j, y) in row_terms[i]))
        ii = @expression(model, sum(imag(y) * vr[j] + real(y) * vi[j] for (j, y) in row_terms[i]))

        p_inj = @expression(model, vr[i] * ir + vi[i] * ii)
        q_inj = @expression(model, vi[i] * ir - vr[i] * ii)

        p_gen = isempty(node_gen_map[i]) ? 0.0 : @expression(model, sum(coeff * Pg[g] for (g, coeff) in node_gen_map[i]))
        q_gen = isempty(node_gen_map[i]) ? 0.0 : @expression(model, sum(coeff * Qg[g] for (g, coeff) in node_gen_map[i]))

        @constraint(model, p_inj == p_gen - p_load[i])
        @constraint(model, q_inj == q_gen - q_load[i])
    end

    slack_indices = [ybus.all_index[bp] for bp in ybus.slack_order]
    @constraint(model, Pg_sub == sum(
        begin
            irs = sum(real(y) * vr[j] - imag(y) * vi[j] for (j, y) in row_terms[sidx])
            iis = sum(imag(y) * vr[j] + real(y) * vi[j] for (j, y) in row_terms[sidx])
            vr[sidx] * irs + vi[sidx] * iis
        end
        for sidx in slack_indices
    ))
    @constraint(model, Qg_sub == sum(
        begin
            irs = sum(real(y) * vr[j] - imag(y) * vi[j] for (j, y) in row_terms[sidx])
            iis = sum(imag(y) * vr[j] + real(y) * vi[j] for (j, y) in row_terms[sidx])
            vi[sidx] * irs - vr[sidx] * iis
        end
        for sidx in slack_indices
    ))

    delta_max = deg2rad(delta_deg)
    tan_delta = tan(delta_max)

    for line in network.lines
        if line.is_switch && !line.is_closed
            continue
        end

        z_series = complex.(line.rmatrix, line.xmatrix) * line.length
        y_series = try
            inv(z_series) / network.base.Ybase
        catch
            continue
        end

        imax = max(line.normamps, line.emergamps)
        if !(isfinite(imax) && imax > 0.0)
            continue
        end
        imax_eff = current_limit_scale * imax

        for r in eachindex(line.phases)
            ph_r = line.phases[r]
            bp_fr = FeederFlow.BusPhase(line.from.bus, ph_r)
            bp_tr = FeederFlow.BusPhase(line.to.bus, ph_r)
            if !(haskey(ybus.all_index, bp_fr) && haskey(ybus.all_index, bp_tr))
                continue
            end

            i_f = ybus.all_index[bp_fr]
            i_t = ybus.all_index[bp_tr]
            dot = @expression(model, vr[i_f] * vr[i_t] + vi[i_f] * vi[i_t])
            cross = @expression(model, vr[i_f] * vi[i_t] - vi[i_f] * vr[i_t])
            @constraint(model, dot >= 0.0)
            @constraint(model, cross <= tan_delta * dot)
            @constraint(model, -cross <= tan_delta * dot)

            ir_line = @expression(model, sum(
                begin
                    ph_c = line.phases[c]
                    bp_fc = FeederFlow.BusPhase(line.from.bus, ph_c)
                    bp_tc = FeederFlow.BusPhase(line.to.bus, ph_c)
                    if !(haskey(ybus.all_index, bp_fc) && haskey(ybus.all_index, bp_tc))
                        0.0
                    else
                        j_f = ybus.all_index[bp_fc]
                        j_t = ybus.all_index[bp_tc]
                        yrc = y_series[r, c]
                        real(yrc) * (vr[j_f] - vr[j_t]) - imag(yrc) * (vi[j_f] - vi[j_t])
                    end
                end
                for c in eachindex(line.phases)
            ))

            ii_line = @expression(model, sum(
                begin
                    ph_c = line.phases[c]
                    bp_fc = FeederFlow.BusPhase(line.from.bus, ph_c)
                    bp_tc = FeederFlow.BusPhase(line.to.bus, ph_c)
                    if !(haskey(ybus.all_index, bp_fc) && haskey(ybus.all_index, bp_tc))
                        0.0
                    else
                        j_f = ybus.all_index[bp_fc]
                        j_t = ybus.all_index[bp_tc]
                        yrc = y_series[r, c]
                        imag(yrc) * (vr[j_f] - vr[j_t]) + real(yrc) * (vi[j_f] - vi[j_t])
                    end
                end
                for c in eachindex(line.phases)
            ))

            @constraint(model, ir_line^2 + ii_line^2 <= imax_eff^2)
        end
    end

    gen_obj = ngen > 0 ?
        5.0 * Pg_sub^2 + 50.0 * Pg_sub + sum(0.5 * gen_specs[g].cost * Pg[g]^2 + gen_specs[g].cost * Pg[g] for g in 1:ngen) :
        5.0 * Pg_sub^2 + 50.0 * Pg_sub

    v_dev = @expression(model, sum((vr[i] - start_vr[i])^2 + (vi[i] - start_vi[i])^2 for i in 1:nall))

    @objective(model, Min, gen_obj + 1e2 * v_dev)

    optimize!(model)

    return (
        network = network,
        ybus = ybus,
        row_terms = row_terms,
        model = model,
        vr = vr,
        vi = vi,
        Pg = Pg,
        Qg = Qg,
        Pg_sub = Pg_sub,
        Qg_sub = Qg_sub,
        gen_specs = gen_specs,
        node_gen_map = node_gen_map,
        p_load = p_load,
        q_load = q_load,
        delta_max = delta_max,
        current_limit_scale = current_limit_scale,
        s_sub_max = s_sub_max,
        load_model = load_model,
        noload = noload,
    )
end

function build_network_without_generators(network::FeederFlow.NetworkModel)
    empty_gens = FeederFlow.ComponentTable(FeederFlow.GeneratorDevice[])
    return FeederFlow.NetworkModel(
        network.buses,
        network.slack_bus,
        network.source,
        network.linecodes,
        network.lines,
        network.transformers,
        network.regulators,
        network.capacitors,
        empty_gens,
        network.loads,
        network.base,
        network.provenance,
    )
end

function compute_balance_mismatch(ybus::FeederFlow.YBusModel, load_model::FeederFlow.LoadModel, v_all::Vector{ComplexF64}, s_demand::Vector{ComplexF64})
    nnet = length(ybus.network_order)
    Yeff = copy(ybus.Ynet)
    Yeff[1:nnet, 1:nnet] .+= load_model.YL
    row_terms = build_sparse_row_terms(Yeff)

    max_p = 0.0
    max_q = 0.0
    for i in 1:nnet
        inode = 0.0 + 0.0im
        for (j, y) in row_terms[i]
            inode += y * v_all[j]
        end
        s_net = v_all[i] * conj(inode)
        s_mis = s_net + s_demand[i]
        max_p = max(max_p, abs(real(s_mis)))
        max_q = max(max_q, abs(imag(s_mis)))
    end
    return max_p, max_q
end

function targeted_power_balance_test(dss_path::AbstractString)
    network = parse_network_with_fallback(dss_path)
    loads_only_network = build_network_without_generators(network)
    bundle = FeederFlow.solve_power_flow(loads_only_network; method = :zbus, max_iter = 80, tol = 1e-8)

    ybus = bundle.ybus
    nnet = length(ybus.network_order)
    v_all = bundle.result.voltages
    v_net = v_all[1:nnet]

    i_exact = FeederFlow.load_currents(bundle.loads, v_net)
    s_exact = v_net .* conj.(i_exact)

    p_alloc, q_alloc = allocate_loads_to_nodes(loads_only_network, ybus)
    s_alloc = complex.(p_alloc[1:nnet], q_alloc[1:nnet])

    node_err = abs.(s_exact .- s_alloc)
    max_node_err = isempty(node_err) ? 0.0 : maximum(node_err)
    mean_node_err = isempty(node_err) ? 0.0 : sum(node_err) / length(node_err)

    max_p_exact, max_q_exact = compute_balance_mismatch(ybus, bundle.loads, v_all, s_exact)
    max_p_alloc, max_q_alloc = compute_balance_mismatch(ybus, bundle.loads, v_all, s_alloc)

    model_counts = Dict{Int,Int}()
    delta_count = 0
    for ld in network.loads
        model_counts[ld.model] = get(model_counts, ld.model, 0) + 1
        ld.conn == :delta && (delta_count += 1)
    end

    return (
        pf_converged = bundle.result.converged,
        pf_iters = bundle.result.iterations,
        model_counts = model_counts,
        delta_loads = delta_count,
        total_loads = length(network.loads),
        total_exact_p = sum(real.(s_exact)),
        total_exact_q = sum(imag.(s_exact)),
        total_alloc_p = sum(real.(s_alloc)),
        total_alloc_q = sum(imag.(s_alloc)),
        max_node_power_error = max_node_err,
        mean_node_power_error = mean_node_err,
        exact_max_p_mismatch = max_p_exact,
        exact_max_q_mismatch = max_q_exact,
        alloc_max_p_mismatch = max_p_alloc,
        alloc_max_q_mismatch = max_q_alloc,
    )
end

targeted_power_balance_test (generic function with 1 method)

In [83]:
function line_constraint_data_quality(network::FeederFlow.NetworkModel)
    usable = 0
    skipped_singular = 0
    skipped_nonfinite_z = 0
    skipped_nonfinite_y = 0
    skipped_no_imax = 0

    for line in network.lines
        if line.is_switch && !line.is_closed
            continue
        end

        imax = max(line.normamps, line.emergamps)
        if !(isfinite(imax) && imax > 0.0)
            skipped_no_imax += 1
            continue
        end

        z_series = complex.(line.rmatrix, line.xmatrix) * line.length
        z_ok = all(isfinite.(real.(z_series))) && all(isfinite.(imag.(z_series)))
        if !z_ok
            skipped_nonfinite_z += 1
            continue
        end

        y_series = try
            inv(z_series) / network.base.Ybase
        catch
            skipped_singular += 1
            continue
        end

        y_ok = all(isfinite.(real.(y_series))) && all(isfinite.(imag.(y_series)))
        if !y_ok
            skipped_nonfinite_y += 1
            continue
        end

        usable += 1
    end

    return (
        usable = usable,
        skipped_singular = skipped_singular,
        skipped_nonfinite_z = skipped_nonfinite_z,
        skipped_nonfinite_y = skipped_nonfinite_y,
        skipped_no_imax = skipped_no_imax,
    )
end

function exact_pf_nodal_demands(network::FeederFlow.NetworkModel, ybus::FeederFlow.YBusModel)
    loads_only_network = build_network_without_generators(network)
    bundle = FeederFlow.solve_power_flow(loads_only_network; method = :zbus, max_iter = 80, tol = 1e-8)

    nall = length(ybus.all_order)
    nnet = length(ybus.network_order)
    p_load = zeros(Float64, nall)
    q_load = zeros(Float64, nall)

    v_net = bundle.result.voltages[1:nnet]
    i_exact = FeederFlow.load_currents(bundle.loads, v_net)
    s_exact = v_net .* conj.(i_exact)
    for i in 1:nnet
        p_load[i] = real(s_exact[i])
        q_load[i] = imag(s_exact[i])
    end

    return p_load, q_load, bundle
end

function compute_nodal_balance_mismatch(row_terms, vrv, viv, p_load, q_load, node_gen_map, pg_vals, qg_vals, nnet)
    max_p = 0.0
    max_q = 0.0
    for i in 1:nnet
        ir = 0.0
        ii = 0.0
        for (j, y) in row_terms[i]
            ir += real(y) * vrv[j] - imag(y) * viv[j]
            ii += imag(y) * vrv[j] + real(y) * viv[j]
        end
        p_lhs = vrv[i] * ir + viv[i] * ii
        q_lhs = viv[i] * ir - vrv[i] * ii

        p_gen = 0.0
        q_gen = 0.0
        for (g, coeff) in node_gen_map[i]
            p_gen += coeff * pg_vals[g]
            q_gen += coeff * qg_vals[g]
        end

        max_p = max(max_p, abs(p_lhs - (p_gen - p_load[i])))
        max_q = max(max_q, abs(q_lhs - (q_gen - q_load[i])))
    end
    return max_p, max_q
end

function solve_power_balance_only_feasibility(
    dss_path::AbstractString;
    demand_mode::Symbol = :allocated,
    enforce_voltage_bounds::Bool = true,
    enforce_substation_limit::Bool = true,
    substation_scale::Float64 = 1.0,
    remove_generators::Bool = false,
    objective_mode::Symbol = :economic,
    use_limited_memory::Bool = false,
)
    network_raw = parse_network_with_fallback(dss_path)
    network = remove_generators ? build_network_without_generators(network_raw) : network_raw
    ybus = FeederFlow.build_y(network)

    nall = length(ybus.all_order)
    nnet = length(ybus.network_order)

    noload = FeederFlow.compute_no_load(ybus; v_slack = FeederFlow.source_slack(network.source, network.base))
    load_model = FeederFlow.build_load_model(network, ybus, noload)
    Yeff = copy(ybus.Ynet)
    Yeff[1:nnet, 1:nnet] .+= load_model.YL
    row_terms = build_sparse_row_terms(Yeff)

    exact_start_map = Dict{FeederFlow.BusPhase,ComplexF64}()
    p_load, q_load = if demand_mode == :allocated
        allocate_loads_to_nodes(network, ybus)
    elseif demand_mode == :pf_exact
        p_tmp, q_tmp, exact_bundle = exact_pf_nodal_demands(network, ybus)
        exact_start_map = exact_bundle.result.phase_voltages
        p_tmp, q_tmp
    else
        error("Unsupported demand_mode=$demand_mode")
    end

    gen_specs, node_gen_map = build_generator_specs(network, ybus)
    ngen = length(gen_specs)

    vmin = fill(0.90, nall)
    vmax = fill(1.10, nall)
    for i in 1:nnet
        bp = ybus.all_order[i]
        vscale = FeederFlow.bus_vbase(network.base, bp.bus) / network.base.Vbase
        if haskey(network.buses, bp.bus)
            bus = network.buses[bp.bus]
            vmin[i] = bus.vmin_pu * vscale
            vmax[i] = bus.vmax_pu * vscale
        else
            vmin[i] *= vscale
            vmax[i] *= vscale
        end
    end

    slack_phasors = build_balanced_slack_phasors(network.source)

    function start_map_is_usable(map::Dict{FeederFlow.BusPhase,ComplexF64})
        isempty(map) && return false
        for (bp, v) in map
            local_mag = abs(v) * (network.base.Vbase / FeederFlow.bus_vbase(network.base, bp.bus))
            if !(isfinite(real(v)) && isfinite(imag(v)) && isfinite(local_mag) && local_mag <= 2.5)
                return false
            end
        end
        return true
    end

    start_map = Dict{FeederFlow.BusPhase,ComplexF64}()
    start_source = "default_slack"
    if start_map_is_usable(exact_start_map)
        start_map = exact_start_map
        start_source = "exact_pf"
    end
    if isempty(start_map)
        try
            pf_bundle_loads = FeederFlow.solve_power_flow(build_network_without_generators(network); method = :zbus, max_iter = 80, tol = 1e-8)
            candidate = pf_bundle_loads.result.phase_voltages
            if start_map_is_usable(candidate)
                start_map = candidate
                start_source = "loads_pf"
            end
        catch
        end
    end
    if isempty(start_map)
        try
            pf_bundle = FeederFlow.solve_power_flow(network; method = :zbus, max_iter = 80, tol = 1e-8)
            candidate = pf_bundle.result.phase_voltages
            if start_map_is_usable(candidate)
                start_map = candidate
                start_source = "network_pf"
            end
        catch
        end
    end

    start_vr = zeros(Float64, nall)
    start_vi = zeros(Float64, nall)
    for i in 1:nall
        bp = ybus.all_order[i]
        v_start = get(start_map, bp, get(slack_phasors, bp.phase, network.source.pu + 0im))
        start_vr[i] = real(v_start)
        start_vi[i] = imag(v_start)
    end

    pg_start = zeros(Float64, ngen)
    qg_start = zeros(Float64, ngen)
    start_max_p_mismatch, start_max_q_mismatch = compute_nodal_balance_mismatch(row_terms, start_vr, start_vi, p_load, q_load, node_gen_map, pg_start, qg_start, nnet)
    slack_indices = [ybus.all_index[bp] for bp in ybus.slack_order]
    start_pg_sub = sum(
        begin
            irs = sum(real(y) * start_vr[j] - imag(y) * start_vi[j] for (j, y) in row_terms[sidx])
            iis = sum(imag(y) * start_vr[j] + real(y) * start_vi[j] for (j, y) in row_terms[sidx])
            start_vr[sidx] * irs + start_vi[sidx] * iis
        end
        for sidx in slack_indices
    )
    start_qg_sub = sum(
        begin
            irs = sum(real(y) * start_vr[j] - imag(y) * start_vi[j] for (j, y) in row_terms[sidx])
            iis = sum(imag(y) * start_vr[j] + real(y) * start_vi[j] for (j, y) in row_terms[sidx])
            start_vi[sidx] * irs - start_vr[sidx] * iis
        end
        for sidx in slack_indices
    )

    model = JuMP.Model(Ipopt.Optimizer)
    set_silent(model)
    set_optimizer_attribute(model, "tol", 1e-8)
    set_optimizer_attribute(model, "max_iter", 4000)
    if use_limited_memory
        set_optimizer_attribute(model, "hessian_approximation", "limited-memory")
    end

    vbox = zeros(Float64, nall)
    for i in 1:nall
        bp = ybus.all_order[i]
        vbox[i] = 2.5 * FeederFlow.bus_vbase(network.base, bp.bus) / network.base.Vbase
    end

    @variable(model, vr[i=1:nall], lower_bound = -vbox[i], upper_bound = vbox[i])
    @variable(model, vi[i=1:nall], lower_bound = -vbox[i], upper_bound = vbox[i])

    for i in 1:nall
        set_start_value(vr[i], start_vr[i])
        set_start_value(vi[i], start_vi[i])
    end

    for bp in ybus.slack_order
        idx = ybus.all_index[bp]
        vref = get(slack_phasors, bp.phase, network.source.pu + 0im)
        @constraint(model, vr[idx] == real(vref))
        @constraint(model, vi[idx] == imag(vref))
    end

    if enforce_voltage_bounds
        for i in 1:nnet
            @constraint(model, vmin[i]^2 <= vr[i]^2 + vi[i]^2)
            @constraint(model, vr[i]^2 + vi[i]^2 <= vmax[i]^2)
        end
    end

    Pg = JuMP.VariableRef[]
    Qg = JuMP.VariableRef[]
    if ngen > 0
        @variable(model, 0.0 <= Pg_var[g=1:ngen] <= gen_specs[g].pmax)
        @variable(model, gen_specs[g].qmin <= Qg_var[g=1:ngen] <= gen_specs[g].qmax)
        Pg = collect(Pg_var)
        Qg = collect(Qg_var)
        for g in 1:ngen
            @constraint(model, Pg[g]^2 + Qg[g]^2 <= gen_specs[g].smax^2)
            set_start_value(Pg[g], 0.0)
            set_start_value(Qg[g], 0.0)
        end
    end

    s_sub_max = substation_scale * estimate_substation_smax(network)
    @variable(model, -s_sub_max <= Pg_sub <= s_sub_max)
    @variable(model, -s_sub_max <= Qg_sub <= s_sub_max)
    if enforce_substation_limit
        @constraint(model, Pg_sub^2 + Qg_sub^2 <= s_sub_max^2)
    end

    for i in 1:nnet
        ir = @expression(model, sum(real(y) * vr[j] - imag(y) * vi[j] for (j, y) in row_terms[i]))
        ii = @expression(model, sum(imag(y) * vr[j] + real(y) * vi[j] for (j, y) in row_terms[i]))

        p_inj = @expression(model, vr[i] * ir + vi[i] * ii)
        q_inj = @expression(model, vi[i] * ir - vr[i] * ii)

        p_gen = isempty(node_gen_map[i]) ? 0.0 : @expression(model, sum(coeff * Pg[g] for (g, coeff) in node_gen_map[i]))
        q_gen = isempty(node_gen_map[i]) ? 0.0 : @expression(model, sum(coeff * Qg[g] for (g, coeff) in node_gen_map[i]))

        @constraint(model, p_inj == p_gen - p_load[i])
        @constraint(model, q_inj == q_gen - q_load[i])
    end

    @constraint(model, Pg_sub == sum(
        begin
            irs = sum(real(y) * vr[j] - imag(y) * vi[j] for (j, y) in row_terms[sidx])
            iis = sum(imag(y) * vr[j] + real(y) * vi[j] for (j, y) in row_terms[sidx])
            vr[sidx] * irs + vi[sidx] * iis
        end
        for sidx in slack_indices
    ))
    @constraint(model, Qg_sub == sum(
        begin
            irs = sum(real(y) * vr[j] - imag(y) * vi[j] for (j, y) in row_terms[sidx])
            iis = sum(imag(y) * vr[j] + real(y) * vi[j] for (j, y) in row_terms[sidx])
            vi[sidx] * irs - vr[sidx] * iis
        end
        for sidx in slack_indices
    ))

    v_dev = @expression(model, sum((vr[i] - start_vr[i])^2 + (vi[i] - start_vi[i])^2 for i in 1:nall))
    if objective_mode == :feasibility
        @objective(model, Min, v_dev)
    else
        @objective(model, Min, 1e2 * v_dev + Pg_sub^2)
    end

    optimize!(model)

    term = termination_status(model)
    prim = primal_status(model)
    raw = raw_status(model)
    hasv = has_values(model)

    if !hasv
        return (
            term = term,
            prim = prim,
            raw = raw,
            has_values = false,
            objective = NaN,
            vm_min = NaN,
            vm_max = NaN,
            max_p_mismatch = NaN,
            max_q_mismatch = NaN,
            start_max_p_mismatch = start_max_p_mismatch,
            start_max_q_mismatch = start_max_q_mismatch,
            start_pg_sub = start_pg_sub,
            start_qg_sub = start_qg_sub,
            start_source = start_source,
            s_sub_max = s_sub_max,
            pg_sub = NaN,
            qg_sub = NaN,
        )
    end

    vrv = value.(vr)
    viv = value.(vi)
    vm = Float64[]
    for i in 1:nnet
        bp = ybus.all_order[i]
        vmag_sys = sqrt(vrv[i]^2 + viv[i]^2)
        vmag_local = vmag_sys * (network.base.Vbase / FeederFlow.bus_vbase(network.base, bp.bus))
        push!(vm, vmag_local)
    end

    pg_vals = ngen == 0 ? Float64[] : [value(Pg[g]) for g in 1:ngen]
    qg_vals = ngen == 0 ? Float64[] : [value(Qg[g]) for g in 1:ngen]

    max_p_mis, max_q_mis = compute_nodal_balance_mismatch(row_terms, vrv, viv, p_load, q_load, node_gen_map, pg_vals, qg_vals, nnet)

    return (
        term = term,
        prim = prim,
        raw = raw,
        has_values = true,
        objective = objective_value(model),
        vm_min = minimum(vm),
        vm_max = maximum(vm),
        max_p_mismatch = max_p_mis,
        max_q_mismatch = max_q_mis,
        start_max_p_mismatch = start_max_p_mismatch,
        start_max_q_mismatch = start_max_q_mismatch,
        start_pg_sub = start_pg_sub,
        start_qg_sub = start_qg_sub,
        start_source = start_source,
        s_sub_max = s_sub_max,
        pg_sub = value(Pg_sub),
        qg_sub = value(Qg_sub),
    )
end

solve_power_balance_only_feasibility (generic function with 1 method)

In [73]:
function evaluate_opf_solution(art)
    model = art.model
    network = art.network
    ybus = art.ybus
    row_terms = art.row_terms

    term = termination_status(model)
    prim = primal_status(model)

    if !has_values(model)
        return (
            term = term,
            prim = prim,
            has_values = false,
            objective = NaN,
            vm_min = NaN,
            vm_max = NaN,
            pg_sub = NaN,
            qg_sub = NaN,
            max_p_mismatch = NaN,
            max_q_mismatch = NaN,
            max_angle_usage = NaN,
            max_current_usage = NaN,
            sub_s_usage = NaN,
        )
    end

    vrv = value.(art.vr)
    viv = value.(art.vi)
    nnet = length(ybus.network_order)

    pgen = zeros(Float64, length(vrv))
    qgen = zeros(Float64, length(vrv))
    for i in eachindex(art.node_gen_map)
        for (g, coeff) in art.node_gen_map[i]
            pgen[i] += coeff * value(art.Pg[g])
            qgen[i] += coeff * value(art.Qg[g])
        end
    end

    max_p_mismatch = 0.0
    max_q_mismatch = 0.0
    for i in 1:nnet
        ir = 0.0
        ii = 0.0
        for (j, y) in row_terms[i]
            ir += real(y) * vrv[j] - imag(y) * viv[j]
            ii += imag(y) * vrv[j] + real(y) * viv[j]
        end
        p_lhs = vrv[i] * ir + viv[i] * ii
        q_lhs = viv[i] * ir - vrv[i] * ii
        p_rhs = pgen[i] - art.p_load[i]
        q_rhs = qgen[i] - art.q_load[i]
        max_p_mismatch = max(max_p_mismatch, abs(p_lhs - p_rhs))
        max_q_mismatch = max(max_q_mismatch, abs(q_lhs - q_rhs))
    end

    max_angle_usage = 0.0
    max_current_usage = 0.0
    for line in network.lines
        if line.is_switch && !line.is_closed
            continue
        end

        z_series = complex.(line.rmatrix, line.xmatrix) * line.length
        y_series = try
            inv(z_series) / network.base.Ybase
        catch
            continue
        end

        imax = max(line.normamps, line.emergamps)
        if !(isfinite(imax) && imax > 0.0)
            continue
        end
        imax_eff = art.current_limit_scale * imax

        for r in eachindex(line.phases)
            ph_r = line.phases[r]
            bp_fr = FeederFlow.BusPhase(line.from.bus, ph_r)
            bp_tr = FeederFlow.BusPhase(line.to.bus, ph_r)
            if !(haskey(ybus.all_index, bp_fr) && haskey(ybus.all_index, bp_tr))
                continue
            end

            i_f = ybus.all_index[bp_fr]
            i_t = ybus.all_index[bp_tr]
            vf = complex(vrv[i_f], viv[i_f])
            vt = complex(vrv[i_t], viv[i_t])
            dtheta = atan(sin(angle(vf) - angle(vt)), cos(angle(vf) - angle(vt)))
            max_angle_usage = max(max_angle_usage, abs(dtheta) / art.delta_max)

            i_line = 0.0 + 0.0im
            for c in eachindex(line.phases)
                ph_c = line.phases[c]
                bp_fc = FeederFlow.BusPhase(line.from.bus, ph_c)
                bp_tc = FeederFlow.BusPhase(line.to.bus, ph_c)
                if !(haskey(ybus.all_index, bp_fc) && haskey(ybus.all_index, bp_tc))
                    continue
                end
                j_f = ybus.all_index[bp_fc]
                j_t = ybus.all_index[bp_tc]
                vdiff = complex(vrv[j_f] - vrv[j_t], viv[j_f] - viv[j_t])
                i_line += y_series[r, c] * vdiff
            end
            max_current_usage = max(max_current_usage, abs(i_line) / imax_eff)
        end
    end

    vm = Float64[]
    for i in 1:nnet
        bp = ybus.all_order[i]
        vmag_sys = sqrt(vrv[i]^2 + viv[i]^2)
        vmag_local = vmag_sys * (network.base.Vbase / FeederFlow.bus_vbase(network.base, bp.bus))
        push!(vm, vmag_local)
    end
    sub_s_usage = hypot(value(art.Pg_sub), value(art.Qg_sub)) / art.s_sub_max

    return (
        term = term,
        prim = prim,
        has_values = true,
        objective = objective_value(model),
        vm_min = minimum(vm),
        vm_max = maximum(vm),
        pg_sub = value(art.Pg_sub),
        qg_sub = value(art.Qg_sub),
        max_p_mismatch = max_p_mismatch,
        max_q_mismatch = max_q_mismatch,
        max_angle_usage = max_angle_usage,
        max_current_usage = max_current_usage,
        sub_s_usage = sub_s_usage,
    )
end

function recommended_solutions(pb)
    actions = String[]

    if pb.max_node_power_error > 1e-3 || pb.alloc_max_p_mismatch > 1e-3 || pb.alloc_max_q_mismatch > 1e-3
        push!(actions, "Replace equal-share nodal load allocation with branch-pair load equations built from FeederFlow LoadContribution data.")
    end

    if get(pb.model_counts, 3, 0) > 0 || get(pb.model_counts, 4, 0) > 0 || get(pb.model_counts, 5, 0) > 0
        push!(actions, "Add explicit nonlinear load terms for model=3/4/5 in OPF power balance instead of constant-PQ approximation.")
    end

    if pb.delta_loads > 0
        push!(actions, "Model delta loads as line-to-line branch powers instead of splitting power equally across node phases.")
    end

    isempty(actions) && push!(actions, "Power-balance mapping is consistent; investigate branch-angle/current limits and source bounds next.")
    return actions
end

feeder_cases = [
    ("IEEE13", joinpath(@__DIR__, "..", "examples", "grids", "13_bus", "IEEE13Nodeckt.dss")),
    ("IEEE37", joinpath(@__DIR__, "..", "examples", "grids", "37_bus", "ieee37.dss")),
    ("IEEE123", joinpath(@__DIR__, "..", "examples", "grids", "123_bus", "IEEE123Master.dss")),
]

power_balance_runs = Dict{String,Any}()
opf_runs = Dict{String,Any}()

for (name, path) in feeder_cases
    @printf("\n=== %s: Targeted Power-Balance Tests ===\n", name)
    pb = targeted_power_balance_test(path)
    power_balance_runs[name] = pb

    @printf("pf converged              = %s (iters=%d)\n", string(pb.pf_converged), pb.pf_iters)
    @printf("load model counts         = m1:%d m2:%d m3:%d m4:%d m5:%d\n",
        get(pb.model_counts, 1, 0),
        get(pb.model_counts, 2, 0),
        get(pb.model_counts, 3, 0),
        get(pb.model_counts, 4, 0),
        get(pb.model_counts, 5, 0),
    )
    @printf("delta loads / total       = %d / %d\n", pb.delta_loads, pb.total_loads)
    @printf("total load exact P,Q      = %.6f, %.6f\n", pb.total_exact_p, pb.total_exact_q)
    @printf("total load alloc P,Q      = %.6f, %.6f\n", pb.total_alloc_p, pb.total_alloc_q)
    @printf("max node |S exact-alloc|  = %.3e\n", pb.max_node_power_error)
    @printf("mean node |S exact-alloc| = %.3e\n", pb.mean_node_power_error)
    @printf("exact max |P/Q mismatch|  = %.3e / %.3e\n", pb.exact_max_p_mismatch, pb.exact_max_q_mismatch)
    @printf("alloc max |P/Q mismatch|  = %.3e / %.3e\n", pb.alloc_max_p_mismatch, pb.alloc_max_q_mismatch)

    fixes = recommended_solutions(pb)
    for (k, msg) in enumerate(fixes)
        @printf("solution %d                = %s\n", k, msg)
    end

    @printf("\n=== %s: No-Slack OPF ===\n", name)
    art = build_and_solve_feeder_opf(path)
    metrics = evaluate_opf_solution(art)
    opf_runs[name] = (art = art, metrics = metrics)

    @printf("termination_status        = %s\n", string(metrics.term))
    @printf("primal_status             = %s\n", string(metrics.prim))
    @printf("has_values                = %s\n", string(metrics.has_values))

    if metrics.has_values
        @printf("objective                 = %.8f\n", metrics.objective)
        @printf("substation Pg, Qg         = %.6f, %.6f\n", metrics.pg_sub, metrics.qg_sub)
        @printf("min/max Vm (network nodes)= %.6f / %.6f pu\n", metrics.vm_min, metrics.vm_max)
        @printf("max |P mismatch|          = %.3e\n", metrics.max_p_mismatch)
        @printf("max |Q mismatch|          = %.3e\n", metrics.max_q_mismatch)
        @printf("max angle usage           = %.4f pu of limit\n", metrics.max_angle_usage)
        @printf("max current usage         = %.4f pu of limit\n", metrics.max_current_usage)
        @printf("substation S usage        = %.4f pu of limit\n", metrics.sub_s_usage)
    end
end


=== IEEE13: Targeted Power-Balance Tests ===
pf converged              = true (iters=10)
load model counts         = m1:11 m2:2 m3:0 m4:0 m5:2
delta loads / total       = 3 / 15
total load exact P,Q      = 0.568143, 0.345324
total load alloc P,Q      = 0.621600, 0.376800
max node |S exact-alloc|  = 1.486e-02
mean node |S exact-alloc| = 2.386e-03
exact max |P/Q mismatch|  = 2.209e-09 / 6.016e-10
alloc max |P/Q mismatch|  = 1.383e-02 / 1.045e-02
solution 1                = Replace equal-share nodal load allocation with branch-pair load equations built from FeederFlow LoadContribution data.
solution 2                = Add explicit nonlinear load terms for model=3/4/5 in OPF power balance instead of constant-PQ approximation.
solution 3                = Model delta loads as line-to-line branch powers instead of splitting power equally across node phases.

=== IEEE13: No-Slack OPF ===
termination_status        = LOCALLY_INFEASIBLE
primal_status             = INFEASIBLE_POINT
has_values    

In [78]:
function print_pb_variant(label::AbstractString, r)
    @printf("%s status = %s | %s | raw=%s\n", label, string(r.term), string(r.prim), r.raw)
    if r.has_values
        @printf("%s mismatch = %.3e / %.3e | Vm=%.6f..%.6f | Pg_sub,Qg_sub=%.6f,%.6f\n",
            label,
            r.max_p_mismatch,
            r.max_q_mismatch,
            r.vm_min,
            r.vm_max,
            r.pg_sub,
            r.qg_sub,
        )
    end
    @printf("%s start mismatch = %.3e / %.3e\n", label, r.start_max_p_mismatch, r.start_max_q_mismatch)
end

deep_diagnostics = Dict{String,Any}()

for (name, path) in feeder_cases
    network = parse_network_with_fallback(path)
    ybus = FeederFlow.build_y(network)
    yvals = nonzeros(ybus.Ynet)
    y_max = isempty(yvals) ? 0.0 : maximum(abs.(yvals))
    y_min = isempty(yvals) ? 0.0 : minimum(abs.(yvals))

    pf_network = FeederFlow.solve_power_flow(network; method = :zbus, max_iter = 80, tol = 1e-8)
    nnet = length(ybus.network_order)
    pf_vm = Float64[]
    for i in 1:nnet
        bp = ybus.all_order[i]
        vmag_sys = abs(pf_network.result.voltages[i])
        vmag_local = vmag_sys * (network.base.Vbase / FeederFlow.bus_vbase(network.base, bp.bus))
        push!(pf_vm, vmag_local)
    end

    line_q = line_constraint_data_quality(network)

    pb_alloc = solve_power_balance_only_feasibility(path; demand_mode = :allocated)
    pb_exact = solve_power_balance_only_feasibility(path; demand_mode = :pf_exact)
    pb_exact_feas = solve_power_balance_only_feasibility(path; demand_mode = :pf_exact, objective_mode = :feasibility)
    pb_exact_loose_sub = solve_power_balance_only_feasibility(path; demand_mode = :pf_exact, substation_scale = 20.0)
    pb_exact_no_sub = solve_power_balance_only_feasibility(path; demand_mode = :pf_exact, enforce_substation_limit = false)
    pb_exact_no_v_no_sub = solve_power_balance_only_feasibility(path; demand_mode = :pf_exact, enforce_voltage_bounds = false, enforce_substation_limit = false)
    pb_exact_no_gen = solve_power_balance_only_feasibility(path; demand_mode = :pf_exact, remove_generators = true)

    relaxed_art = build_and_solve_feeder_opf(path; delta_deg = 89.0, current_limit_scale = 1.0e6)
    relaxed_metrics = evaluate_opf_solution(relaxed_art)

    deep_diagnostics[name] = (
        y_max = y_max,
        y_min = y_min,
        pf_vm_min = minimum(pf_vm),
        pf_vm_max = maximum(pf_vm),
        line_quality = line_q,
        pb_alloc = pb_alloc,
        pb_exact = pb_exact,
        pb_exact_feas = pb_exact_feas,
        pb_exact_loose_sub = pb_exact_loose_sub,
        pb_exact_no_sub = pb_exact_no_sub,
        pb_exact_no_v_no_sub = pb_exact_no_v_no_sub,
        pb_exact_no_gen = pb_exact_no_gen,
        full_relaxed = relaxed_metrics,
    )

    @printf("\n=== %s: Deep Diagnostic Ablation ===\n", name)
    @printf("Ynet |min|max              = %.3e / %.3e\n", y_min, y_max)
    @printf("PF Vm range                = %.6f / %.6f\n", minimum(pf_vm), maximum(pf_vm))
    @printf("line quality usable/singular/nonfinite_z/nonfinite_y/no_imax = %d / %d / %d / %d / %d\n",
        line_q.usable,
        line_q.skipped_singular,
        line_q.skipped_nonfinite_z,
        line_q.skipped_nonfinite_y,
        line_q.skipped_no_imax,
    )

    print_pb_variant("PB alloc", pb_alloc)
    print_pb_variant("PB exact", pb_exact)
    print_pb_variant("PB exact feasibility-obj", pb_exact_feas)
    print_pb_variant("PB exact loose-sub", pb_exact_loose_sub)
    print_pb_variant("PB exact no-sub", pb_exact_no_sub)
    print_pb_variant("PB exact no-v/no-sub", pb_exact_no_v_no_sub)
    print_pb_variant("PB exact no-gen", pb_exact_no_gen)

    @printf("Full relaxed status        = %s | %s\n", string(relaxed_metrics.term), string(relaxed_metrics.prim))
    if relaxed_metrics.has_values
        @printf("Full relaxed mismatch      = %.3e / %.3e\n", relaxed_metrics.max_p_mismatch, relaxed_metrics.max_q_mismatch)
        @printf("Full relaxed angle/current = %.4f / %.4f\n", relaxed_metrics.max_angle_usage, relaxed_metrics.max_current_usage)
    end
end


=== IEEE13: Deep Diagnostic Ablation ===
Ynet |min|max              = 3.909e-10 / 1.154e+07
PF Vm range                = 0.921000 / 1.000077
line quality usable/singular/nonfinite_z/nonfinite_y/no_imax = 11 / 0 / 0 / 0 / 1
PB alloc status = LOCALLY_INFEASIBLE | INFEASIBLE_POINT | raw=Infeasible_Problem_Detected
PB alloc mismatch = 7.623e-01 / 5.681e-01 | Vm=0.000000..1.100000 | Pg_sub,Qg_sub=1.082969,1.686237
PB alloc start mismatch = 1.383e-02 / 1.045e-02
PB exact status = LOCALLY_INFEASIBLE | INFEASIBLE_POINT | raw=Infeasible_Problem_Detected
PB exact mismatch = 9.318e-01 / 5.743e-01 | Vm=0.035567..1.100000 | Pg_sub,Qg_sub=0.019289,-0.477983
PB exact start mismatch = 2.209e-09 / 6.016e-10
PB exact feasibility-obj status = LOCALLY_INFEASIBLE | INFEASIBLE_POINT | raw=Infeasible_Problem_Detected
PB exact feasibility-obj mismatch = 7.643e-01 / 5.576e-01 | Vm=0.000000..1.100000 | Pg_sub,Qg_sub=1.082720,1.686900
PB exact feasibility-obj start mismatch = 2.209e-09 / 6.016e-10
PB exact loos

In [52]:
println("\nCondensed deep-diagnostic summary")
for (name, d) in sort(collect(deep_diagnostics); by = x -> x[1])
    pa = d.pb_alloc
    pe = d.pb_exact
    ps = d.pb_exact_loose_sub
    pn = d.pb_exact_no_sub
    pv = d.pb_exact_no_v_no_sub
    pg = d.pb_exact_no_gen

    @printf("%s | PF Vm %.4f..%.4f | Ymax %.3e\n", name, d.pf_vm_min, d.pf_vm_max, d.y_max)
    @printf("  alloc term/raw: %s / %s | mis %.2e/%.2e | start %.2e/%.2e\n", string(pa.term), pa.raw, pa.max_p_mismatch, pa.max_q_mismatch, pa.start_max_p_mismatch, pa.start_max_q_mismatch)
    @printf("  exact term/raw: %s / %s | mis %.2e/%.2e | start %.2e/%.2e\n", string(pe.term), pe.raw, pe.max_p_mismatch, pe.max_q_mismatch, pe.start_max_p_mismatch, pe.start_max_q_mismatch)
    @printf("  exact loose-sub: %s / %s | mis %.2e/%.2e | start %.2e/%.2e\n", string(ps.term), ps.raw, ps.max_p_mismatch, ps.max_q_mismatch, ps.start_max_p_mismatch, ps.start_max_q_mismatch)
    @printf("  exact no-sub:    %s / %s | mis %.2e/%.2e | start %.2e/%.2e\n", string(pn.term), pn.raw, pn.max_p_mismatch, pn.max_q_mismatch, pn.start_max_p_mismatch, pn.start_max_q_mismatch)
    @printf("  exact no-v/no-sub:%s / %s | mis %.2e/%.2e | start %.2e/%.2e\n", string(pv.term), pv.raw, pv.max_p_mismatch, pv.max_q_mismatch, pv.start_max_p_mismatch, pv.start_max_q_mismatch)
    @printf("  exact no-gen:    %s / %s | mis %.2e/%.2e | start %.2e/%.2e\n", string(pg.term), pg.raw, pg.max_p_mismatch, pg.max_q_mismatch, pg.start_max_p_mismatch, pg.start_max_q_mismatch)
end


Condensed deep-diagnostic summary
IEEE123 | PF Vm 0.9877..1.0561 | Ymax 1.154e+06
  alloc term/raw: OTHER_ERROR / Restoration_Failed | mis 3.11e-10/5.86e-10 | start 2.69e-03/2.79e-03
  exact term/raw: ALMOST_LOCALLY_SOLVED / Solved_To_Acceptable_Level | mis 1.63e-04/4.29e-05 | start 7.60e-10/2.50e-10
  exact loose-sub: NUMERICAL_ERROR / Error_In_Step_Computation | mis 4.84e-10/2.90e-10 | start 7.60e-10/2.50e-10
  exact no-sub:    ALMOST_LOCALLY_SOLVED / Solved_To_Acceptable_Level | mis 1.62e-04/4.30e-05 | start 7.60e-10/2.50e-10
  exact no-v/no-sub:OTHER_ERROR / Restoration_Failed | mis 5.29e-10/3.44e-10 | start 7.60e-10/2.50e-10
  exact no-gen:    ALMOST_LOCALLY_SOLVED / Solved_To_Acceptable_Level | mis 2.54e-04/3.66e-05 | start 7.60e-10/2.50e-10
IEEE13 | PF Vm 0.9210..1.0001 | Ymax 1.154e+07
  alloc term/raw: LOCALLY_INFEASIBLE / Infeasible_Problem_Detected | mis 9.80e-01/8.62e-01 | start 5.71e+02/3.97e+03
  exact term/raw: LOCALLY_INFEASIBLE / Infeasible_Problem_Detected | mis 9.84

In [39]:
println("\nY-bus ordering consistency check")
for (name, path) in feeder_cases
    network = parse_network_with_fallback(path)
    y_with_gen = FeederFlow.build_y(network)
    y_no_gen = FeederFlow.build_y(build_network_without_generators(network))

    same_len = length(y_with_gen.all_order) == length(y_no_gen.all_order)
    same_all = same_len && all(y_with_gen.all_order .== y_no_gen.all_order)
    same_net = length(y_with_gen.network_order) == length(y_no_gen.network_order) && all(y_with_gen.network_order .== y_no_gen.network_order)

    @printf("%s | all_order same = %s | network_order same = %s\n", name, string(same_all), string(same_net))
end


Y-bus ordering consistency check
IEEE13 | all_order same = true | network_order same = true
IEEE37 | all_order same = true | network_order same = true
IEEE123 | all_order same = true | network_order same = true


In [57]:
println("DIAG_COMPACT")
for name in sort(collect(keys(deep_diagnostics)))
    d = deep_diagnostics[name]
    pb = d.pb_exact
    png = d.pb_exact_no_gen
    full = d.full_relaxed
    @printf("%s|pb=%s/%s|mis=%.3e/%.3e|start=%.3e/%.3e|src=%s|no_gen=%s/%s|full=%s/%s\n",
        name,
        string(pb.term),
        pb.raw,
        pb.max_p_mismatch,
        pb.max_q_mismatch,
        pb.start_max_p_mismatch,
        pb.start_max_q_mismatch,
        pb.start_source,
        string(png.term),
        png.raw,
        string(full.term),
        string(full.prim),
    )
end

DIAG_COMPACT
IEEE123|pb=ALMOST_LOCALLY_SOLVED/Solved_To_Acceptable_Level|mis=1.629e-04/4.295e-05|start=7.603e-10/2.500e-10|src=exact_pf|no_gen=ALMOST_LOCALLY_SOLVED/Solved_To_Acceptable_Level|full=ITERATION_LIMIT/INFEASIBLE_POINT
IEEE13|pb=LOCALLY_INFEASIBLE/Infeasible_Problem_Detected|mis=9.838e-01/8.770e-01|start=5.711e+02/3.966e+03|src=default_slack|no_gen=LOCALLY_INFEASIBLE/Infeasible_Problem_Detected|full=LOCALLY_INFEASIBLE/INFEASIBLE_POINT
IEEE37|pb=LOCALLY_INFEASIBLE/Infeasible_Problem_Detected|mis=1.393e+00/4.434e+00|start=1.644e+01/3.307e+02|src=default_slack|no_gen=LOCALLY_INFEASIBLE/Infeasible_Problem_Detected|full=LOCALLY_INFEASIBLE/INFEASIBLE_POINT


In [76]:
for name in ["IEEE13", "IEEE37", "IEEE123"]
    d = deep_diagnostics[name]
    pb = d.pb_exact
    println(name, " src=", pb.start_source,
        " pb=", string(pb.term), "/", pb.raw,
        " mis=", @sprintf("%.3e/%.3e", pb.max_p_mismatch, pb.max_q_mismatch),
        " start=", @sprintf("%.3e/%.3e", pb.start_max_p_mismatch, pb.start_max_q_mismatch),
        " full=", string(d.full_relaxed.term), "/", string(d.full_relaxed.prim)
    )
end

IEEE13 src=exact_pf pb=LOCALLY_INFEASIBLE/Infeasible_Problem_Detected mis=9.318e-01/5.743e-01 start=2.209e-09/6.016e-10 full=LOCALLY_INFEASIBLE/INFEASIBLE_POINT
IEEE37 src=exact_pf pb=LOCALLY_INFEASIBLE/Infeasible_Problem_Detected mis=1.294e+00/4.458e+00 start=1.047e-08/2.860e-10 full=LOCALLY_INFEASIBLE/INFEASIBLE_POINT
IEEE123 src=exact_pf pb=NUMERICAL_ERROR/Error_In_Step_Computation mis=6.832e-10/2.294e-10 start=7.603e-10/2.500e-10 full=ALMOST_LOCALLY_SOLVED/NEARLY_FEASIBLE_POINT


In [62]:
function map_quality(map::Dict{FeederFlow.BusPhase,ComplexF64}, ybus::FeederFlow.YBusModel)
    if isempty(map)
        return (keys = 0, missing = length(ybus.all_order), maxabs = NaN, nonfinite = 0, gt2 = 0)
    end
    maxabs = maximum(abs.(collect(values(map))))
    nonfinite = count(v -> !(isfinite(real(v)) && isfinite(imag(v))), values(map))
    gt2 = count(v -> abs(v) > 2.0, values(map))
    missing = count(bp -> !haskey(map, bp), ybus.all_order)
    return (keys = length(map), missing = missing, maxabs = maxabs, nonfinite = nonfinite, gt2 = gt2)
end

for (name, path) in feeder_cases
    network = parse_network_with_fallback(path)
    ybus = FeederFlow.build_y(network)

    exact_bundle = exact_pf_nodal_demands(network, ybus)[3]
    q_exact = map_quality(exact_bundle.result.phase_voltages, ybus)

    q_loads = begin
        bundle = FeederFlow.solve_power_flow(build_network_without_generators(network); method = :zbus, max_iter = 80, tol = 1e-8)
        map_quality(bundle.result.phase_voltages, ybus)
    end

    q_net = begin
        bundle = FeederFlow.solve_power_flow(network; method = :zbus, max_iter = 80, tol = 1e-8)
        map_quality(bundle.result.phase_voltages, ybus)
    end

    @printf("%s exact(keys=%d miss=%d max=%.3e nf=%d gt2=%d) loads(keys=%d miss=%d max=%.3e nf=%d gt2=%d) net(keys=%d miss=%d max=%.3e nf=%d gt2=%d)\n",
        name,
        q_exact.keys, q_exact.missing, q_exact.maxabs, q_exact.nonfinite, q_exact.gt2,
        q_loads.keys, q_loads.missing, q_loads.maxabs, q_loads.nonfinite, q_loads.gt2,
        q_net.keys, q_net.missing, q_net.maxabs, q_net.nonfinite, q_net.gt2,
    )
end

IEEE13 exact(keys=41 miss=0 max=2.765e+01 nf=0 gt2=3) loads(keys=41 miss=0 max=2.765e+01 nf=0 gt2=3) net(keys=41 miss=0 max=2.765e+01 nf=0 gt2=3)
IEEE37 exact(keys=117 miss=0 max=4.792e+01 nf=0 gt2=3) loads(keys=117 miss=0 max=4.792e+01 nf=0 gt2=3) net(keys=117 miss=0 max=3.658e+161 nf=0 gt2=117)
IEEE123 exact(keys=274 miss=0 max=1.050e+00 nf=0 gt2=0) loads(keys=274 miss=0 max=1.050e+00 nf=0 gt2=0) net(keys=274 miss=0 max=1.056e+00 nf=0 gt2=0)


In [67]:
for name in ["IEEE13", "IEEE37", "IEEE123"]
    d = deep_diagnostics[name]
    b = d.pb_exact
    nsub = d.pb_exact_no_sub
    nvnsub = d.pb_exact_no_v_no_sub
    @printf("%s | exact=%s/%s (%.3e,%.3e) | no_sub=%s/%s (%.3e,%.3e) | no_v_no_sub=%s/%s (%.3e,%.3e)\n",
        name,
        string(b.term), b.raw, b.max_p_mismatch, b.max_q_mismatch,
        string(nsub.term), nsub.raw, nsub.max_p_mismatch, nsub.max_q_mismatch,
        string(nvnsub.term), nvnsub.raw, nvnsub.max_p_mismatch, nvnsub.max_q_mismatch,
    )
end

IEEE13 | exact=LOCALLY_INFEASIBLE/Infeasible_Problem_Detected (9.016e-01,1.333e+00) | no_sub=LOCALLY_INFEASIBLE/Infeasible_Problem_Detected (8.943e-01,1.250e+00) | no_v_no_sub=ITERATION_LIMIT/Maximum_Iterations_Exceeded (3.815e-01,1.233e+00)
IEEE37 | exact=LOCALLY_INFEASIBLE/Infeasible_Problem_Detected (2.536e-01,2.394e-01) | no_sub=LOCALLY_INFEASIBLE/Infeasible_Problem_Detected (2.567e-01,2.395e-01) | no_v_no_sub=LOCALLY_INFEASIBLE/Infeasible_Problem_Detected (3.600e-01,2.160e-01)
IEEE123 | exact=ALMOST_LOCALLY_SOLVED/Solved_To_Acceptable_Level (1.629e-04,4.295e-05) | no_sub=ALMOST_LOCALLY_SOLVED/Solved_To_Acceptable_Level (1.621e-04,4.302e-05) | no_v_no_sub=OTHER_ERROR/Restoration_Failed (5.289e-10,3.437e-10)


In [79]:
for name in ["IEEE13", "IEEE37", "IEEE123"]
    d = deep_diagnostics[name]
    econ = d.pb_exact
    feas = d.pb_exact_feas
    @printf("%s econ=%s/%s (%.3e,%.3e) feas=%s/%s (%.3e,%.3e) start=%.3e/%.3e src=%s start_sub=%.3e/%.3e lim=%.3e\n",
        name,
        string(econ.term), econ.raw, econ.max_p_mismatch, econ.max_q_mismatch,
        string(feas.term), feas.raw, feas.max_p_mismatch, feas.max_q_mismatch,
        econ.start_max_p_mismatch, econ.start_max_q_mismatch, econ.start_source,
        econ.start_pg_sub, econ.start_qg_sub, econ.s_sub_max,
    )
end

IEEE13 econ=LOCALLY_INFEASIBLE/Infeasible_Problem_Detected (9.318e-01,5.743e-01) feas=LOCALLY_INFEASIBLE/Infeasible_Problem_Detected (7.643e-01,5.576e-01) start=2.209e-09/6.016e-10 src=exact_pf start_sub=6.528e-01/3.190e-01 lim=4.254e+00
IEEE37 econ=LOCALLY_INFEASIBLE/Infeasible_Problem_Detected (1.294e+00,4.458e+00) feas=LOCALLY_INFEASIBLE/Infeasible_Problem_Detected (2.534e-01,2.394e-01) start=1.047e-08/2.860e-10 src=exact_pf start_sub=8.672e-01/5.018e-01 lim=5.670e+00
IEEE123 econ=NUMERICAL_ERROR/Error_In_Step_Computation (6.832e-10,2.294e-10) feas=LOCALLY_SOLVED/Solve_Succeeded (5.163e-10,2.852e-10) start=7.603e-10/2.500e-10 src=exact_pf start_sub=7.231e-01/2.623e-01 lim=4.183e+00


In [80]:
for name in ["IEEE13", "IEEE37", "IEEE123"]
    ng = deep_diagnostics[name].pb_exact_no_gen
    @printf("%s no_gen=%s/%s mis=%.3e/%.3e start=%.3e/%.3e src=%s start_sub=%.3e/%.3e lim=%.3e\n",
        name,
        string(ng.term), ng.raw,
        ng.max_p_mismatch, ng.max_q_mismatch,
        ng.start_max_p_mismatch, ng.start_max_q_mismatch,
        ng.start_source,
        ng.start_pg_sub, ng.start_qg_sub, ng.s_sub_max,
    )
end

IEEE13 no_gen=LOCALLY_INFEASIBLE/Infeasible_Problem_Detected mis=8.538e-01/6.077e-01 start=2.209e-09/6.016e-10 src=exact_pf start_sub=6.528e-01/3.190e-01 lim=4.254e+00
IEEE37 no_gen=LOCALLY_INFEASIBLE/Infeasible_Problem_Detected mis=1.591e+00/4.560e+00 start=1.047e-08/2.860e-10 src=exact_pf start_sub=8.672e-01/5.018e-01 lim=5.670e+00
IEEE123 no_gen=ALMOST_LOCALLY_SOLVED/Solved_To_Acceptable_Level mis=5.053e-10/2.706e-10 start=7.603e-10/2.500e-10 src=exact_pf start_sub=7.231e-01/2.623e-01 lim=4.183e+00


In [81]:
function solve_pb_scaled_exact_nogen(dss_path::AbstractString)
    network_raw = parse_network_with_fallback(dss_path)
    network = build_network_without_generators(network_raw)
    ybus = FeederFlow.build_y(network)

    nall = length(ybus.all_order)
    nnet = length(ybus.network_order)

    noload = FeederFlow.compute_no_load(ybus; v_slack = FeederFlow.source_slack(network.source, network.base))
    load_model = FeederFlow.build_load_model(network, ybus, noload)
    Yeff = copy(ybus.Ynet)
    Yeff[1:nnet, 1:nnet] .+= load_model.YL
    row_terms = build_sparse_row_terms(Yeff)

    p_load, q_load, exact_bundle = exact_pf_nodal_demands(network, ybus)
    start_map = exact_bundle.result.phase_voltages

    alpha = zeros(Float64, nall)
    start_xr = zeros(Float64, nall)
    start_xi = zeros(Float64, nall)
    for i in 1:nall
        bp = ybus.all_order[i]
        alpha[i] = FeederFlow.bus_vbase(network.base, bp.bus) / network.base.Vbase
        v = get(start_map, bp, network.source.pu + 0im)
        start_xr[i] = real(v) / alpha[i]
        start_xi[i] = imag(v) / alpha[i]
    end

    model = JuMP.Model(Ipopt.Optimizer)
    set_silent(model)
    set_optimizer_attribute(model, "tol", 1e-8)
    set_optimizer_attribute(model, "max_iter", 6000)

    @variable(model, -2.5 <= xr[1:nall] <= 2.5)
    @variable(model, -2.5 <= xi[1:nall] <= 2.5)

    for i in 1:nall
        set_start_value(xr[i], start_xr[i])
        set_start_value(xi[i], start_xi[i])
    end

    slack_phasors = build_balanced_slack_phasors(network.source)
    for bp in ybus.slack_order
        idx = ybus.all_index[bp]
        vref = get(slack_phasors, bp.phase, network.source.pu + 0im)
        @constraint(model, xr[idx] == real(vref) / alpha[idx])
        @constraint(model, xi[idx] == imag(vref) / alpha[idx])
    end

    for i in 1:nnet
        bp = ybus.all_order[i]
        vmin_local = haskey(network.buses, bp.bus) ? network.buses[bp.bus].vmin_pu : 0.9
        vmax_local = haskey(network.buses, bp.bus) ? network.buses[bp.bus].vmax_pu : 1.1
        @constraint(model, vmin_local^2 <= xr[i]^2 + xi[i]^2)
        @constraint(model, xr[i]^2 + xi[i]^2 <= vmax_local^2)
    end

    for i in 1:nnet
        vr_i = @expression(model, alpha[i] * xr[i])
        vi_i = @expression(model, alpha[i] * xi[i])

        ir = @expression(model, sum(real(y) * (alpha[j] * xr[j]) - imag(y) * (alpha[j] * xi[j]) for (j, y) in row_terms[i]))
        ii = @expression(model, sum(imag(y) * (alpha[j] * xr[j]) + real(y) * (alpha[j] * xi[j]) for (j, y) in row_terms[i]))

        p_inj = @expression(model, vr_i * ir + vi_i * ii)
        q_inj = @expression(model, vi_i * ir - vr_i * ii)

        @constraint(model, p_inj == -p_load[i])
        @constraint(model, q_inj == -q_load[i])
    end

    @objective(model, Min, sum((xr[i] - start_xr[i])^2 + (xi[i] - start_xi[i])^2 for i in 1:nall))

    optimize!(model)

    term = termination_status(model)
    prim = primal_status(model)
    raw = raw_status(model)
    if !has_values(model)
        return (term = term, prim = prim, raw = raw, max_p = NaN, max_q = NaN)
    end

    xrv = value.(xr)
    xiv = value.(xi)
    vrv = alpha .* xrv
    viv = alpha .* xiv

    max_p, max_q = compute_nodal_balance_mismatch(row_terms, vrv, viv, p_load, q_load, [Tuple{Int,Float64}[] for _ in 1:nall], Float64[], Float64[], nnet)
    return (term = term, prim = prim, raw = raw, max_p = max_p, max_q = max_q)
end

for (name, path) in [
    ("IEEE13", joinpath(@__DIR__, "..", "examples", "grids", "13_bus", "IEEE13Nodeckt.dss")),
    ("IEEE37", joinpath(@__DIR__, "..", "examples", "grids", "37_bus", "ieee37.dss")),
]
    r = solve_pb_scaled_exact_nogen(path)
    @printf("%s scaled-no-gen => %s / %s / %s | mis %.3e / %.3e\n", name, string(r.term), string(r.prim), r.raw, r.max_p, r.max_q)
end

IEEE13 scaled-no-gen => LOCALLY_INFEASIBLE / INFEASIBLE_POINT / Infeasible_Problem_Detected | mis 1.993e-01 / 4.597e-01
IEEE37 scaled-no-gen => LOCALLY_INFEASIBLE / INFEASIBLE_POINT / Infeasible_Problem_Detected | mis 1.591e+00 / 4.560e+00


In [82]:
function check_start_feasibility_nogen(dss_path::AbstractString)
    network = build_network_without_generators(parse_network_with_fallback(dss_path))
    ybus = FeederFlow.build_y(network)
    nall = length(ybus.all_order)
    nnet = length(ybus.network_order)

    noload = FeederFlow.compute_no_load(ybus; v_slack = FeederFlow.source_slack(network.source, network.base))
    load_model = FeederFlow.build_load_model(network, ybus, noload)
    Yeff = copy(ybus.Ynet)
    Yeff[1:nnet, 1:nnet] .+= load_model.YL
    row_terms = build_sparse_row_terms(Yeff)

    p_load, q_load, exact_bundle = exact_pf_nodal_demands(network, ybus)
    start_map = exact_bundle.result.phase_voltages

    start_vr = zeros(Float64, nall)
    start_vi = zeros(Float64, nall)
    vm_local = zeros(Float64, nnet)

    for i in 1:nall
        bp = ybus.all_order[i]
        v = start_map[bp]
        start_vr[i] = real(v)
        start_vi[i] = imag(v)
        if i <= nnet
            vmag_sys = abs(v)
            vm_local[i] = vmag_sys * (network.base.Vbase / FeederFlow.bus_vbase(network.base, bp.bus))
        end
    end

    start_p, start_q = compute_nodal_balance_mismatch(row_terms, start_vr, start_vi, p_load, q_load, [Tuple{Int,Float64}[] for _ in 1:nall], Float64[], Float64[], nnet)

    vmin_viol = 0.0
    vmax_viol = 0.0
    for i in 1:nnet
        bp = ybus.all_order[i]
        vmin_local = haskey(network.buses, bp.bus) ? network.buses[bp.bus].vmin_pu : 0.9
        vmax_local = haskey(network.buses, bp.bus) ? network.buses[bp.bus].vmax_pu : 1.1
        vmin_viol = max(vmin_viol, max(0.0, vmin_local - vm_local[i]))
        vmax_viol = max(vmax_viol, max(0.0, vm_local[i] - vmax_local))
    end

    return (start_p = start_p, start_q = start_q, vm_min = minimum(vm_local), vm_max = maximum(vm_local), vmin_viol = vmin_viol, vmax_viol = vmax_viol)
end

for (name, path) in [
    ("IEEE13", joinpath(@__DIR__, "..", "examples", "grids", "13_bus", "IEEE13Nodeckt.dss")),
    ("IEEE37", joinpath(@__DIR__, "..", "examples", "grids", "37_bus", "ieee37.dss")),
]
    s = check_start_feasibility_nogen(path)
    @printf("%s start_check: mis %.3e/%.3e vm %.6f..%.6f vmin_viol %.3e vmax_viol %.3e\n", name, s.start_p, s.start_q, s.vm_min, s.vm_max, s.vmin_viol, s.vmax_viol)
end

IEEE13 start_check: mis 2.209e-09/6.016e-10 vm 0.913684..1.000075 vmin_viol 0.000e+00 vmax_viol 0.000e+00
IEEE37 start_check: mis 1.047e-08/2.860e-10 vm 0.883580..0.951785 vmin_viol 1.642e-02 vmax_viol 0.000e+00


In [84]:
for (name, path) in [
    ("IEEE13", joinpath(@__DIR__, "..", "examples", "grids", "13_bus", "IEEE13Nodeckt.dss")),
    ("IEEE37", joinpath(@__DIR__, "..", "examples", "grids", "37_bus", "ieee37.dss")),
]
    base = solve_power_balance_only_feasibility(path; demand_mode = :pf_exact, remove_generators = true, objective_mode = :feasibility)
    lm = solve_power_balance_only_feasibility(path; demand_mode = :pf_exact, remove_generators = true, objective_mode = :feasibility, use_limited_memory = true)
    @printf("%s no-gen feas: base=%s/%s (%.3e,%.3e), lm=%s/%s (%.3e,%.3e)\n",
        name,
        string(base.term), base.raw, base.max_p_mismatch, base.max_q_mismatch,
        string(lm.term), lm.raw, lm.max_p_mismatch, lm.max_q_mismatch,
    )
end

IEEE13 no-gen feas: base=LOCALLY_INFEASIBLE/Infeasible_Problem_Detected (2.150e+00,5.390e+00), lm=ITERATION_LIMIT/Maximum_Iterations_Exceeded (8.285e-01,5.600e-01)
IEEE37 no-gen feas: base=LOCALLY_INFEASIBLE/Infeasible_Problem_Detected (4.141e-01,3.429e-01), lm=LOCALLY_INFEASIBLE/Infeasible_Problem_Detected (1.496e+00,4.719e+00)


In [87]:
function solve_power_balance_two_stage(
    dss_path::AbstractString;
    demand_mode::Symbol = :pf_exact,
    enforce_voltage_bounds::Bool = true,
    enforce_substation_limit::Bool = true,
    substation_scale::Float64 = 1.0,
    remove_generators::Bool = false,
    use_limited_memory::Bool = true,
    stage2_objective_mode::Symbol = :economic,
    stage1_slack_weight::Float64 = 1e6,
)
    network_raw = parse_network_with_fallback(dss_path)
    network = remove_generators ? build_network_without_generators(network_raw) : network_raw
    ybus = FeederFlow.build_y(network)

    nall = length(ybus.all_order)
    nnet = length(ybus.network_order)

    noload = FeederFlow.compute_no_load(ybus; v_slack = FeederFlow.source_slack(network.source, network.base))
    load_model = FeederFlow.build_load_model(network, ybus, noload)
    Yeff = copy(ybus.Ynet)
    Yeff[1:nnet, 1:nnet] .+= load_model.YL
    row_terms = build_sparse_row_terms(Yeff)

    exact_start_map = Dict{FeederFlow.BusPhase,ComplexF64}()
    p_load, q_load = if demand_mode == :allocated
        allocate_loads_to_nodes(network, ybus)
    elseif demand_mode == :pf_exact
        p_tmp, q_tmp, exact_bundle = exact_pf_nodal_demands(network, ybus)
        exact_start_map = exact_bundle.result.phase_voltages
        p_tmp, q_tmp
    else
        error("Unsupported demand_mode=$demand_mode")
    end

    gen_specs, node_gen_map = build_generator_specs(network, ybus)
    ngen = length(gen_specs)

    vmin = fill(0.90, nall)
    vmax = fill(1.10, nall)
    for i in 1:nnet
        bp = ybus.all_order[i]
        vscale = FeederFlow.bus_vbase(network.base, bp.bus) / network.base.Vbase
        if haskey(network.buses, bp.bus)
            bus = network.buses[bp.bus]
            vmin[i] = bus.vmin_pu * vscale
            vmax[i] = bus.vmax_pu * vscale
        else
            vmin[i] *= vscale
            vmax[i] *= vscale
        end
    end

    slack_phasors = build_balanced_slack_phasors(network.source)

    function start_map_is_usable(map::Dict{FeederFlow.BusPhase,ComplexF64})
        isempty(map) && return false
        for (bp, v) in map
            local_mag = abs(v) * (network.base.Vbase / FeederFlow.bus_vbase(network.base, bp.bus))
            if !(isfinite(real(v)) && isfinite(imag(v)) && isfinite(local_mag) && local_mag <= 2.5)
                return false
            end
        end
        return true
    end

    start_map = Dict{FeederFlow.BusPhase,ComplexF64}()
    start_source = "default_slack"
    if start_map_is_usable(exact_start_map)
        start_map = exact_start_map
        start_source = "exact_pf"
    end
    if isempty(start_map)
        try
            pf_bundle_loads = FeederFlow.solve_power_flow(build_network_without_generators(network); method = :zbus, max_iter = 80, tol = 1e-8)
            candidate = pf_bundle_loads.result.phase_voltages
            if start_map_is_usable(candidate)
                start_map = candidate
                start_source = "loads_pf"
            end
        catch
        end
    end
    if isempty(start_map)
        try
            pf_bundle = FeederFlow.solve_power_flow(network; method = :zbus, max_iter = 80, tol = 1e-8)
            candidate = pf_bundle.result.phase_voltages
            if start_map_is_usable(candidate)
                start_map = candidate
                start_source = "network_pf"
            end
        catch
        end
    end

    start_vr = zeros(Float64, nall)
    start_vi = zeros(Float64, nall)
    for i in 1:nall
        bp = ybus.all_order[i]
        v_start = get(start_map, bp, get(slack_phasors, bp.phase, network.source.pu + 0im))
        start_vr[i] = real(v_start)
        start_vi[i] = imag(v_start)
    end

    start_pg = zeros(Float64, ngen)
    start_qg = zeros(Float64, ngen)
    start_max_p_mismatch, start_max_q_mismatch = compute_nodal_balance_mismatch(
        row_terms,
        start_vr,
        start_vi,
        p_load,
        q_load,
        node_gen_map,
        start_pg,
        start_qg,
        nnet,
    )

    slack_indices = [ybus.all_index[bp] for bp in ybus.slack_order]
    start_pg_sub = sum(
        begin
            irs = sum(real(y) * start_vr[j] - imag(y) * start_vi[j] for (j, y) in row_terms[sidx])
            iis = sum(imag(y) * start_vr[j] + real(y) * start_vi[j] for (j, y) in row_terms[sidx])
            start_vr[sidx] * irs + start_vi[sidx] * iis
        end
        for sidx in slack_indices
    )
    start_qg_sub = sum(
        begin
            irs = sum(real(y) * start_vr[j] - imag(y) * start_vi[j] for (j, y) in row_terms[sidx])
            iis = sum(imag(y) * start_vr[j] + real(y) * start_vi[j] for (j, y) in row_terms[sidx])
            start_vi[sidx] * irs - start_vr[sidx] * iis
        end
        for sidx in slack_indices
    )

    vbox = zeros(Float64, nall)
    for i in 1:nall
        bp = ybus.all_order[i]
        vbox[i] = 2.5 * FeederFlow.bus_vbase(network.base, bp.bus) / network.base.Vbase
    end

    s_sub_max = substation_scale * estimate_substation_smax(network)

    function build_stage_model(
        init_vr::Vector{Float64},
        init_vi::Vector{Float64};
        init_pg::Vector{Float64},
        init_qg::Vector{Float64},
        init_pg_sub::Float64,
        init_qg_sub::Float64,
        allow_balance_slack::Bool,
        objective_mode::Symbol,
    )
        model = JuMP.Model(Ipopt.Optimizer)
        set_silent(model)
        set_optimizer_attribute(model, "tol", 1e-8)
        set_optimizer_attribute(model, "max_iter", 6000)
        if use_limited_memory
            set_optimizer_attribute(model, "hessian_approximation", "limited-memory")
        end

        @variable(model, vr[i=1:nall], lower_bound = -vbox[i], upper_bound = vbox[i])
        @variable(model, vi[i=1:nall], lower_bound = -vbox[i], upper_bound = vbox[i])

        for i in 1:nall
            set_start_value(vr[i], init_vr[i])
            set_start_value(vi[i], init_vi[i])
        end

        for bp in ybus.slack_order
            idx = ybus.all_index[bp]
            vref = get(slack_phasors, bp.phase, network.source.pu + 0im)
            @constraint(model, vr[idx] == real(vref))
            @constraint(model, vi[idx] == imag(vref))
        end

        if enforce_voltage_bounds
            for i in 1:nnet
                @constraint(model, vmin[i]^2 <= vr[i]^2 + vi[i]^2)
                @constraint(model, vr[i]^2 + vi[i]^2 <= vmax[i]^2)
            end
        end

        Pg = JuMP.VariableRef[]
        Qg = JuMP.VariableRef[]
        if ngen > 0
            @variable(model, 0.0 <= Pg_var[g=1:ngen] <= gen_specs[g].pmax)
            @variable(model, gen_specs[g].qmin <= Qg_var[g=1:ngen] <= gen_specs[g].qmax)
            Pg = collect(Pg_var)
            Qg = collect(Qg_var)
            for g in 1:ngen
                @constraint(model, Pg[g]^2 + Qg[g]^2 <= gen_specs[g].smax^2)
                set_start_value(Pg[g], clamp(init_pg[g], 0.0, gen_specs[g].pmax))
                set_start_value(Qg[g], clamp(init_qg[g], gen_specs[g].qmin, gen_specs[g].qmax))
            end
        end

        @variable(model, -s_sub_max <= Pg_sub <= s_sub_max)
        @variable(model, -s_sub_max <= Qg_sub <= s_sub_max)
        if enforce_substation_limit
            @constraint(model, Pg_sub^2 + Qg_sub^2 <= s_sub_max^2)
        end
        set_start_value(Pg_sub, clamp(init_pg_sub, -s_sub_max, s_sub_max))
        set_start_value(Qg_sub, clamp(init_qg_sub, -s_sub_max, s_sub_max))

        p_sp = JuMP.VariableRef[]
        p_sn = JuMP.VariableRef[]
        q_sp = JuMP.VariableRef[]
        q_sn = JuMP.VariableRef[]
        if allow_balance_slack
            @variable(model, p_sp_var[1:nnet] >= 0.0)
            @variable(model, p_sn_var[1:nnet] >= 0.0)
            @variable(model, q_sp_var[1:nnet] >= 0.0)
            @variable(model, q_sn_var[1:nnet] >= 0.0)
            p_sp = collect(p_sp_var)
            p_sn = collect(p_sn_var)
            q_sp = collect(q_sp_var)
            q_sn = collect(q_sn_var)
        end

        for i in 1:nnet
            ir = @expression(model, sum(real(y) * vr[j] - imag(y) * vi[j] for (j, y) in row_terms[i]))
            ii = @expression(model, sum(imag(y) * vr[j] + real(y) * vi[j] for (j, y) in row_terms[i]))

            p_inj = @expression(model, vr[i] * ir + vi[i] * ii)
            q_inj = @expression(model, vi[i] * ir - vr[i] * ii)

            p_gen = isempty(node_gen_map[i]) ? 0.0 : @expression(model, sum(coeff * Pg[g] for (g, coeff) in node_gen_map[i]))
            q_gen = isempty(node_gen_map[i]) ? 0.0 : @expression(model, sum(coeff * Qg[g] for (g, coeff) in node_gen_map[i]))

            if allow_balance_slack
                @constraint(model, p_inj + p_sp[i] - p_sn[i] == p_gen - p_load[i])
                @constraint(model, q_inj + q_sp[i] - q_sn[i] == q_gen - q_load[i])
            else
                @constraint(model, p_inj == p_gen - p_load[i])
                @constraint(model, q_inj == q_gen - q_load[i])
            end
        end

        @constraint(model, Pg_sub == sum(
            begin
                irs = sum(real(y) * vr[j] - imag(y) * vi[j] for (j, y) in row_terms[sidx])
                iis = sum(imag(y) * vr[j] + real(y) * vi[j] for (j, y) in row_terms[sidx])
                vr[sidx] * irs + vi[sidx] * iis
            end
            for sidx in slack_indices
        ))
        @constraint(model, Qg_sub == sum(
            begin
                irs = sum(real(y) * vr[j] - imag(y) * vi[j] for (j, y) in row_terms[sidx])
                iis = sum(imag(y) * vr[j] + real(y) * vi[j] for (j, y) in row_terms[sidx])
                vi[sidx] * irs - vr[sidx] * iis
            end
            for sidx in slack_indices
        ))

        v_dev = @expression(model, sum((vr[i] - init_vr[i])^2 + (vi[i] - init_vi[i])^2 for i in 1:nall))

        if objective_mode == :feasibility
            if allow_balance_slack
                slack_sum = @expression(model, sum(p_sp) + sum(p_sn) + sum(q_sp) + sum(q_sn))
                @objective(model, Min, stage1_slack_weight * slack_sum + v_dev)
            else
                @objective(model, Min, v_dev)
            end
        elseif objective_mode == :economic
            gen_obj = ngen > 0 ?
                5.0 * Pg_sub^2 + 50.0 * Pg_sub + sum(0.5 * gen_specs[g].cost * Pg[g]^2 + gen_specs[g].cost * Pg[g] for g in 1:ngen) :
                5.0 * Pg_sub^2 + 50.0 * Pg_sub
            if allow_balance_slack
                slack_sum = @expression(model, sum(p_sp) + sum(p_sn) + sum(q_sp) + sum(q_sn))
                @objective(model, Min, stage1_slack_weight * slack_sum + 1e2 * v_dev + gen_obj)
            else
                @objective(model, Min, 1e2 * v_dev + gen_obj)
            end
        else
            error("Unsupported objective_mode=$objective_mode")
        end

        return (
            model = model,
            vr = vr,
            vi = vi,
            Pg = Pg,
            Qg = Qg,
            Pg_sub = Pg_sub,
            Qg_sub = Qg_sub,
            p_sp = p_sp,
            p_sn = p_sn,
            q_sp = q_sp,
            q_sn = q_sn,
        )
    end

    function collect_stage_metrics(stage)
        term = termination_status(stage.model)
        prim = primal_status(stage.model)
        raw = raw_status(stage.model)
        if !has_values(stage.model)
            return (
                term = term,
                prim = prim,
                raw = raw,
                has_values = false,
                objective = NaN,
                vm_min = NaN,
                vm_max = NaN,
                max_p_mismatch = NaN,
                max_q_mismatch = NaN,
                pg_sub = NaN,
                qg_sub = NaN,
                balance_slack = NaN,
            )
        end

        vrv = value.(stage.vr)
        viv = value.(stage.vi)
        vm = Float64[]
        for i in 1:nnet
            bp = ybus.all_order[i]
            vmag_sys = sqrt(vrv[i]^2 + viv[i]^2)
            vmag_local = vmag_sys * (network.base.Vbase / FeederFlow.bus_vbase(network.base, bp.bus))
            push!(vm, vmag_local)
        end

        pg_vals = ngen == 0 ? Float64[] : [value(stage.Pg[g]) for g in 1:ngen]
        qg_vals = ngen == 0 ? Float64[] : [value(stage.Qg[g]) for g in 1:ngen]
        max_p_mis, max_q_mis = compute_nodal_balance_mismatch(row_terms, vrv, viv, p_load, q_load, node_gen_map, pg_vals, qg_vals, nnet)

        balance_slack = isempty(stage.p_sp) ? 0.0 : (
            sum(value.(stage.p_sp)) +
            sum(value.(stage.p_sn)) +
            sum(value.(stage.q_sp)) +
            sum(value.(stage.q_sn))
        )

        return (
            term = term,
            prim = prim,
            raw = raw,
            has_values = true,
            objective = objective_value(stage.model),
            vm_min = minimum(vm),
            vm_max = maximum(vm),
            max_p_mismatch = max_p_mis,
            max_q_mismatch = max_q_mis,
            pg_sub = value(stage.Pg_sub),
            qg_sub = value(stage.Qg_sub),
            balance_slack = balance_slack,
        )
    end

    stage1 = build_stage_model(
        start_vr,
        start_vi;
        init_pg = start_pg,
        init_qg = start_qg,
        init_pg_sub = start_pg_sub,
        init_qg_sub = start_qg_sub,
        allow_balance_slack = true,
        objective_mode = :feasibility,
    )
    optimize!(stage1.model)
    stage1_metrics = collect_stage_metrics(stage1)

    stage2_metrics = (
        ran = false,
        term = "NOT_RUN",
        prim = "NOT_RUN",
        raw = "NOT_RUN",
        has_values = false,
        objective = NaN,
        vm_min = NaN,
        vm_max = NaN,
        max_p_mismatch = NaN,
        max_q_mismatch = NaN,
        pg_sub = NaN,
        qg_sub = NaN,
        balance_slack = NaN,
    )

    if stage1_metrics.has_values
        vr1 = value.(stage1.vr)
        vi1 = value.(stage1.vi)
        pg1 = ngen == 0 ? Float64[] : [value(stage1.Pg[g]) for g in 1:ngen]
        qg1 = ngen == 0 ? Float64[] : [value(stage1.Qg[g]) for g in 1:ngen]
        pg_sub1 = value(stage1.Pg_sub)
        qg_sub1 = value(stage1.Qg_sub)

        stage2 = build_stage_model(
            vr1,
            vi1;
            init_pg = pg1,
            init_qg = qg1,
            init_pg_sub = pg_sub1,
            init_qg_sub = qg_sub1,
            allow_balance_slack = false,
            objective_mode = stage2_objective_mode,
        )
        optimize!(stage2.model)
        stage2_core = collect_stage_metrics(stage2)
        stage2_metrics = (
            ran = true,
            term = stage2_core.term,
            prim = stage2_core.prim,
            raw = stage2_core.raw,
            has_values = stage2_core.has_values,
            objective = stage2_core.objective,
            vm_min = stage2_core.vm_min,
            vm_max = stage2_core.vm_max,
            max_p_mismatch = stage2_core.max_p_mismatch,
            max_q_mismatch = stage2_core.max_q_mismatch,
            pg_sub = stage2_core.pg_sub,
            qg_sub = stage2_core.qg_sub,
            balance_slack = stage2_core.balance_slack,
        )
    end

    return (
        start_source = start_source,
        start_max_p_mismatch = start_max_p_mismatch,
        start_max_q_mismatch = start_max_q_mismatch,
        start_pg_sub = start_pg_sub,
        start_qg_sub = start_qg_sub,
        s_sub_max = s_sub_max,
        stage1 = stage1_metrics,
        stage2 = stage2_metrics,
    )
end

function print_two_stage_result(label::AbstractString, run)
    s1 = run.stage1
    s2 = run.stage2

    @printf("%s start src=%s start_mis=%.3e/%.3e start_sub=%.3e/%.3e lim=%.3e\n",
        label,
        run.start_source,
        run.start_max_p_mismatch,
        run.start_max_q_mismatch,
        run.start_pg_sub,
        run.start_qg_sub,
        run.s_sub_max,
    )

    @printf("%s stage1=%s/%s raw=%s hasv=%s slack=%.3e mis=%.3e/%.3e vm=%.6f..%.6f\n",
        label,
        string(s1.term),
        string(s1.prim),
        s1.raw,
        string(s1.has_values),
        s1.balance_slack,
        s1.max_p_mismatch,
        s1.max_q_mismatch,
        s1.vm_min,
        s1.vm_max,
    )

    @printf("%s stage2=%s/%s raw=%s hasv=%s mis=%.3e/%.3e vm=%.6f..%.6f sub=%.3e/%.3e\n",
        label,
        string(s2.term),
        string(s2.prim),
        s2.raw,
        string(s2.has_values),
        s2.max_p_mismatch,
        s2.max_q_mismatch,
        s2.vm_min,
        s2.vm_max,
        s2.pg_sub,
        s2.qg_sub,
    )
end

two_stage_runs = Dict{String,Any}()
for (name, path) in [
    ("IEEE13", joinpath(@__DIR__, "..", "examples", "grids", "13_bus", "IEEE13Nodeckt.dss")),
    ("IEEE37", joinpath(@__DIR__, "..", "examples", "grids", "37_bus", "ieee37.dss")),
]
    with_gen = solve_power_balance_two_stage(path; demand_mode = :pf_exact, remove_generators = false, stage2_objective_mode = :economic)
    no_gen = solve_power_balance_two_stage(path; demand_mode = :pf_exact, remove_generators = true, stage2_objective_mode = :feasibility)

    two_stage_runs[name] = (with_gen = with_gen, no_gen = no_gen)

    print_two_stage_result("$name with_gen", with_gen)
    print_two_stage_result("$name no_gen", no_gen)
end

IEEE13 with_gen start src=exact_pf start_mis=2.209e-09/6.016e-10 start_sub=6.528e-01/3.190e-01 lim=4.254e+00
IEEE13 with_gen stage1=ITERATION_LIMIT/NEARLY_FEASIBLE_POINT raw=Maximum_Iterations_Exceeded hasv=true slack=1.119e+04 mis=5.485e+02/3.413e+03 vm=0.900032..0.922976
IEEE13 with_gen stage2=ITERATION_LIMIT/INFEASIBLE_POINT raw=Maximum_Iterations_Exceeded hasv=true mis=5.636e-01/5.157e-01 vm=0.000007..1.110982 sub=1.083e+00/1.685e+00
IEEE13 no_gen start src=exact_pf start_mis=2.209e-09/6.016e-10 start_sub=6.528e-01/3.190e-01 lim=4.254e+00
IEEE13 no_gen stage1=ITERATION_LIMIT/INFEASIBLE_POINT raw=Maximum_Iterations_Exceeded hasv=true slack=1.119e+04 mis=4.806e+02/3.443e+03 vm=0.900054..0.919313
IEEE13 no_gen stage2=ITERATION_LIMIT/INFEASIBLE_POINT raw=Maximum_Iterations_Exceeded hasv=true mis=9.251e-01/5.076e-01 vm=0.000336..1.101743 sub=1.053e+00/1.543e+00
IEEE37 with_gen start src=exact_pf start_mis=1.047e-08/2.860e-10 start_sub=8.672e-01/5.018e-01 lim=5.670e+00
IEEE37 with_gen st

In [88]:
for lm in (false, true)
    for w in (1e6, 1e8)
        r = solve_power_balance_two_stage(
            joinpath(@__DIR__, "..", "examples", "grids", "13_bus", "IEEE13Nodeckt.dss");
            demand_mode = :pf_exact,
            remove_generators = true,
            use_limited_memory = lm,
            stage2_objective_mode = :feasibility,
            stage1_slack_weight = w,
        )
        @printf("IEEE13 no_gen lm=%s w=%.1e | s1=%s/%s slack=%.3e mis=%.3e/%.3e | s2=%s/%s mis=%.3e/%.3e\n",
            string(lm),
            w,
            string(r.stage1.term),
            string(r.stage1.prim),
            r.stage1.balance_slack,
            r.stage1.max_p_mismatch,
            r.stage1.max_q_mismatch,
            string(r.stage2.term),
            string(r.stage2.prim),
            r.stage2.max_p_mismatch,
            r.stage2.max_q_mismatch,
        )
    end
end

IEEE13 no_gen lm=false w=1.0e+06 | s1=LOCALLY_SOLVED/FEASIBLE_POINT slack=1.119e+04 mis=5.477e+02/3.409e+03 | s2=LOCALLY_INFEASIBLE/INFEASIBLE_POINT mis=8.671e-01/6.104e-01
IEEE13 no_gen lm=false w=1.0e+08 | s1=LOCALLY_SOLVED/FEASIBLE_POINT slack=1.119e+04 mis=5.476e+02/3.409e+03 | s2=LOCALLY_INFEASIBLE/INFEASIBLE_POINT mis=8.348e-01/1.213e+00
IEEE13 no_gen lm=true w=1.0e+06 | s1=ITERATION_LIMIT/INFEASIBLE_POINT slack=1.119e+04 mis=4.806e+02/3.443e+03 | s2=ITERATION_LIMIT/INFEASIBLE_POINT mis=9.251e-01/5.076e-01
IEEE13 no_gen lm=true w=1.0e+08 | s1=ITERATION_LIMIT/INFEASIBLE_POINT slack=1.126e+04 mis=5.433e+02/3.411e+03 | s2=ITERATION_LIMIT/INFEASIBLE_POINT mis=2.269e+00/8.813e-01


In [89]:
function solve_power_balance_with_trust(
    dss_path::AbstractString;
    demand_mode::Symbol = :pf_exact,
    enforce_voltage_bounds::Bool = true,
    enforce_substation_limit::Bool = true,
    substation_scale::Float64 = 1.0,
    remove_generators::Bool = false,
    objective_mode::Symbol = :feasibility,
    use_limited_memory::Bool = true,
    trust_radius::Float64 = Inf,
    preferred_start_map::Union{Nothing,Dict{FeederFlow.BusPhase,ComplexF64}} = nothing,
)
    network_raw = parse_network_with_fallback(dss_path)
    network = remove_generators ? build_network_without_generators(network_raw) : network_raw
    ybus = FeederFlow.build_y(network)

    nall = length(ybus.all_order)
    nnet = length(ybus.network_order)

    noload = FeederFlow.compute_no_load(ybus; v_slack = FeederFlow.source_slack(network.source, network.base))
    load_model = FeederFlow.build_load_model(network, ybus, noload)
    Yeff = copy(ybus.Ynet)
    Yeff[1:nnet, 1:nnet] .+= load_model.YL
    row_terms = build_sparse_row_terms(Yeff)

    exact_start_map = Dict{FeederFlow.BusPhase,ComplexF64}()
    p_load, q_load = if demand_mode == :allocated
        allocate_loads_to_nodes(network, ybus)
    elseif demand_mode == :pf_exact
        p_tmp, q_tmp, exact_bundle = exact_pf_nodal_demands(network, ybus)
        exact_start_map = exact_bundle.result.phase_voltages
        p_tmp, q_tmp
    else
        error("Unsupported demand_mode=$demand_mode")
    end

    gen_specs, node_gen_map = build_generator_specs(network, ybus)
    ngen = length(gen_specs)

    vmin = fill(0.90, nall)
    vmax = fill(1.10, nall)
    for i in 1:nnet
        bp = ybus.all_order[i]
        vscale = FeederFlow.bus_vbase(network.base, bp.bus) / network.base.Vbase
        if haskey(network.buses, bp.bus)
            bus = network.buses[bp.bus]
            vmin[i] = bus.vmin_pu * vscale
            vmax[i] = bus.vmax_pu * vscale
        else
            vmin[i] *= vscale
            vmax[i] *= vscale
        end
    end

    slack_phasors = build_balanced_slack_phasors(network.source)

    function start_map_is_usable(map::Dict{FeederFlow.BusPhase,ComplexF64})
        isempty(map) && return false
        for (bp, v) in map
            local_mag = abs(v) * (network.base.Vbase / FeederFlow.bus_vbase(network.base, bp.bus))
            if !(isfinite(real(v)) && isfinite(imag(v)) && isfinite(local_mag) && local_mag <= 2.5)
                return false
            end
        end
        return true
    end

    start_map = Dict{FeederFlow.BusPhase,ComplexF64}()
    start_source = "default_slack"

    if preferred_start_map !== nothing && start_map_is_usable(preferred_start_map)
        start_map = preferred_start_map
        start_source = "provided"
    elseif start_map_is_usable(exact_start_map)
        start_map = exact_start_map
        start_source = "exact_pf"
    else
        try
            pf_bundle_loads = FeederFlow.solve_power_flow(build_network_without_generators(network); method = :zbus, max_iter = 80, tol = 1e-8)
            candidate = pf_bundle_loads.result.phase_voltages
            if start_map_is_usable(candidate)
                start_map = candidate
                start_source = "loads_pf"
            end
        catch
        end
        if isempty(start_map)
            try
                pf_bundle = FeederFlow.solve_power_flow(network; method = :zbus, max_iter = 80, tol = 1e-8)
                candidate = pf_bundle.result.phase_voltages
                if start_map_is_usable(candidate)
                    start_map = candidate
                    start_source = "network_pf"
                end
            catch
            end
        end
    end

    start_vr = zeros(Float64, nall)
    start_vi = zeros(Float64, nall)
    for i in 1:nall
        bp = ybus.all_order[i]
        v_start = get(start_map, bp, get(slack_phasors, bp.phase, network.source.pu + 0im))
        start_vr[i] = real(v_start)
        start_vi[i] = imag(v_start)
    end

    pg_start = zeros(Float64, ngen)
    qg_start = zeros(Float64, ngen)
    start_max_p_mismatch, start_max_q_mismatch = compute_nodal_balance_mismatch(row_terms, start_vr, start_vi, p_load, q_load, node_gen_map, pg_start, qg_start, nnet)

    slack_indices = [ybus.all_index[bp] for bp in ybus.slack_order]
    start_pg_sub = sum(
        begin
            irs = sum(real(y) * start_vr[j] - imag(y) * start_vi[j] for (j, y) in row_terms[sidx])
            iis = sum(imag(y) * start_vr[j] + real(y) * start_vi[j] for (j, y) in row_terms[sidx])
            start_vr[sidx] * irs + start_vi[sidx] * iis
        end
        for sidx in slack_indices
    )
    start_qg_sub = sum(
        begin
            irs = sum(real(y) * start_vr[j] - imag(y) * start_vi[j] for (j, y) in row_terms[sidx])
            iis = sum(imag(y) * start_vr[j] + real(y) * start_vi[j] for (j, y) in row_terms[sidx])
            start_vi[sidx] * irs - start_vr[sidx] * iis
        end
        for sidx in slack_indices
    )

    model = JuMP.Model(Ipopt.Optimizer)
    set_silent(model)
    set_optimizer_attribute(model, "tol", 1e-8)
    set_optimizer_attribute(model, "max_iter", 6000)
    if use_limited_memory
        set_optimizer_attribute(model, "hessian_approximation", "limited-memory")
    end

    vbox = zeros(Float64, nall)
    for i in 1:nall
        bp = ybus.all_order[i]
        vbox[i] = 2.5 * FeederFlow.bus_vbase(network.base, bp.bus) / network.base.Vbase
    end

    @variable(model, vr[i=1:nall], lower_bound = -vbox[i], upper_bound = vbox[i])
    @variable(model, vi[i=1:nall], lower_bound = -vbox[i], upper_bound = vbox[i])

    for i in 1:nall
        set_start_value(vr[i], start_vr[i])
        set_start_value(vi[i], start_vi[i])
    end

    for bp in ybus.slack_order
        idx = ybus.all_index[bp]
        vref = get(slack_phasors, bp.phase, network.source.pu + 0im)
        @constraint(model, vr[idx] == real(vref))
        @constraint(model, vi[idx] == imag(vref))
    end

    if enforce_voltage_bounds
        for i in 1:nnet
            @constraint(model, vmin[i]^2 <= vr[i]^2 + vi[i]^2)
            @constraint(model, vr[i]^2 + vi[i]^2 <= vmax[i]^2)
        end
    end

    if isfinite(trust_radius)
        for i in 1:nall
            @constraint(model, -trust_radius <= vr[i] - start_vr[i])
            @constraint(model, vr[i] - start_vr[i] <= trust_radius)
            @constraint(model, -trust_radius <= vi[i] - start_vi[i])
            @constraint(model, vi[i] - start_vi[i] <= trust_radius)
        end
    end

    Pg = JuMP.VariableRef[]
    Qg = JuMP.VariableRef[]
    if ngen > 0
        @variable(model, 0.0 <= Pg_var[g=1:ngen] <= gen_specs[g].pmax)
        @variable(model, gen_specs[g].qmin <= Qg_var[g=1:ngen] <= gen_specs[g].qmax)
        Pg = collect(Pg_var)
        Qg = collect(Qg_var)
        for g in 1:ngen
            @constraint(model, Pg[g]^2 + Qg[g]^2 <= gen_specs[g].smax^2)
            set_start_value(Pg[g], 0.0)
            set_start_value(Qg[g], 0.0)
        end
    end

    s_sub_max = substation_scale * estimate_substation_smax(network)
    @variable(model, -s_sub_max <= Pg_sub <= s_sub_max)
    @variable(model, -s_sub_max <= Qg_sub <= s_sub_max)
    if enforce_substation_limit
        @constraint(model, Pg_sub^2 + Qg_sub^2 <= s_sub_max^2)
    end
    set_start_value(Pg_sub, clamp(start_pg_sub, -s_sub_max, s_sub_max))
    set_start_value(Qg_sub, clamp(start_qg_sub, -s_sub_max, s_sub_max))

    for i in 1:nnet
        ir = @expression(model, sum(real(y) * vr[j] - imag(y) * vi[j] for (j, y) in row_terms[i]))
        ii = @expression(model, sum(imag(y) * vr[j] + real(y) * vi[j] for (j, y) in row_terms[i]))

        p_inj = @expression(model, vr[i] * ir + vi[i] * ii)
        q_inj = @expression(model, vi[i] * ir - vr[i] * ii)

        p_gen = isempty(node_gen_map[i]) ? 0.0 : @expression(model, sum(coeff * Pg[g] for (g, coeff) in node_gen_map[i]))
        q_gen = isempty(node_gen_map[i]) ? 0.0 : @expression(model, sum(coeff * Qg[g] for (g, coeff) in node_gen_map[i]))

        @constraint(model, p_inj == p_gen - p_load[i])
        @constraint(model, q_inj == q_gen - q_load[i])
    end

    @constraint(model, Pg_sub == sum(
        begin
            irs = sum(real(y) * vr[j] - imag(y) * vi[j] for (j, y) in row_terms[sidx])
            iis = sum(imag(y) * vr[j] + real(y) * vi[j] for (j, y) in row_terms[sidx])
            vr[sidx] * irs + vi[sidx] * iis
        end
        for sidx in slack_indices
    ))
    @constraint(model, Qg_sub == sum(
        begin
            irs = sum(real(y) * vr[j] - imag(y) * vi[j] for (j, y) in row_terms[sidx])
            iis = sum(imag(y) * vr[j] + real(y) * vi[j] for (j, y) in row_terms[sidx])
            vi[sidx] * irs - vr[sidx] * iis
        end
        for sidx in slack_indices
    ))

    v_dev = @expression(model, sum((vr[i] - start_vr[i])^2 + (vi[i] - start_vi[i])^2 for i in 1:nall))
    if objective_mode == :feasibility
        @objective(model, Min, v_dev)
    elseif objective_mode == :economic
        gen_obj = ngen > 0 ?
            5.0 * Pg_sub^2 + 50.0 * Pg_sub + sum(0.5 * gen_specs[g].cost * Pg[g]^2 + gen_specs[g].cost * Pg[g] for g in 1:ngen) :
            5.0 * Pg_sub^2 + 50.0 * Pg_sub
        @objective(model, Min, 1e2 * v_dev + gen_obj)
    else
        error("Unsupported objective_mode=$objective_mode")
    end

    optimize!(model)

    term = termination_status(model)
    prim = primal_status(model)
    raw = raw_status(model)
    hasv = has_values(model)

    if !hasv
        return (
            term = term,
            prim = prim,
            raw = raw,
            has_values = false,
            objective = NaN,
            vm_min = NaN,
            vm_max = NaN,
            max_p_mismatch = NaN,
            max_q_mismatch = NaN,
            start_max_p_mismatch = start_max_p_mismatch,
            start_max_q_mismatch = start_max_q_mismatch,
            start_pg_sub = start_pg_sub,
            start_qg_sub = start_qg_sub,
            start_source = start_source,
            s_sub_max = s_sub_max,
            pg_sub = NaN,
            qg_sub = NaN,
            solution_map = Dict{FeederFlow.BusPhase,ComplexF64}(),
        )
    end

    vrv = value.(vr)
    viv = value.(vi)
    vm = Float64[]
    for i in 1:nnet
        bp = ybus.all_order[i]
        vmag_sys = sqrt(vrv[i]^2 + viv[i]^2)
        vmag_local = vmag_sys * (network.base.Vbase / FeederFlow.bus_vbase(network.base, bp.bus))
        push!(vm, vmag_local)
    end

    pg_vals = ngen == 0 ? Float64[] : [value(Pg[g]) for g in 1:ngen]
    qg_vals = ngen == 0 ? Float64[] : [value(Qg[g]) for g in 1:ngen]
    max_p_mis, max_q_mis = compute_nodal_balance_mismatch(row_terms, vrv, viv, p_load, q_load, node_gen_map, pg_vals, qg_vals, nnet)

    solution_map = Dict{FeederFlow.BusPhase,ComplexF64}()
    for i in 1:nall
        solution_map[ybus.all_order[i]] = complex(vrv[i], viv[i])
    end

    return (
        term = term,
        prim = prim,
        raw = raw,
        has_values = true,
        objective = objective_value(model),
        vm_min = minimum(vm),
        vm_max = maximum(vm),
        max_p_mismatch = max_p_mis,
        max_q_mismatch = max_q_mis,
        start_max_p_mismatch = start_max_p_mismatch,
        start_max_q_mismatch = start_max_q_mismatch,
        start_pg_sub = start_pg_sub,
        start_qg_sub = start_qg_sub,
        start_source = start_source,
        s_sub_max = s_sub_max,
        pg_sub = value(Pg_sub),
        qg_sub = value(Qg_sub),
        solution_map = solution_map,
    )
end

function solve_power_balance_two_stage_trust(
    dss_path::AbstractString;
    demand_mode::Symbol = :pf_exact,
    enforce_voltage_bounds::Bool = true,
    enforce_substation_limit::Bool = true,
    substation_scale::Float64 = 1.0,
    remove_generators::Bool = false,
    use_limited_memory::Bool = true,
    stage1_trust_radius::Float64 = 5e-4,
    stage2_trust_radius::Float64 = 5e-2,
    stage2_objective_mode::Symbol = :economic,
)
    stage1 = solve_power_balance_with_trust(
        dss_path;
        demand_mode = demand_mode,
        enforce_voltage_bounds = enforce_voltage_bounds,
        enforce_substation_limit = enforce_substation_limit,
        substation_scale = substation_scale,
        remove_generators = remove_generators,
        objective_mode = :feasibility,
        use_limited_memory = use_limited_memory,
        trust_radius = stage1_trust_radius,
        preferred_start_map = nothing,
    )

    stage2 = (
        term = "NOT_RUN",
        prim = "NOT_RUN",
        raw = "NOT_RUN",
        has_values = false,
        objective = NaN,
        vm_min = NaN,
        vm_max = NaN,
        max_p_mismatch = NaN,
        max_q_mismatch = NaN,
        pg_sub = NaN,
        qg_sub = NaN,
    )

    if stage1.has_values
        stage2_core = solve_power_balance_with_trust(
            dss_path;
            demand_mode = demand_mode,
            enforce_voltage_bounds = enforce_voltage_bounds,
            enforce_substation_limit = enforce_substation_limit,
            substation_scale = substation_scale,
            remove_generators = remove_generators,
            objective_mode = stage2_objective_mode,
            use_limited_memory = use_limited_memory,
            trust_radius = stage2_trust_radius,
            preferred_start_map = stage1.solution_map,
        )

        stage2 = (
            term = stage2_core.term,
            prim = stage2_core.prim,
            raw = stage2_core.raw,
            has_values = stage2_core.has_values,
            objective = stage2_core.objective,
            vm_min = stage2_core.vm_min,
            vm_max = stage2_core.vm_max,
            max_p_mismatch = stage2_core.max_p_mismatch,
            max_q_mismatch = stage2_core.max_q_mismatch,
            pg_sub = stage2_core.pg_sub,
            qg_sub = stage2_core.qg_sub,
        )
    end

    return (
        start_source = stage1.start_source,
        start_max_p_mismatch = stage1.start_max_p_mismatch,
        start_max_q_mismatch = stage1.start_max_q_mismatch,
        start_pg_sub = stage1.start_pg_sub,
        start_qg_sub = stage1.start_qg_sub,
        s_sub_max = stage1.s_sub_max,
        stage1 = stage1,
        stage2 = stage2,
    )
end

function print_two_stage_trust_result(label::AbstractString, run)
    s1 = run.stage1
    s2 = run.stage2

    @printf("%s start src=%s start_mis=%.3e/%.3e start_sub=%.3e/%.3e lim=%.3e\n",
        label,
        run.start_source,
        run.start_max_p_mismatch,
        run.start_max_q_mismatch,
        run.start_pg_sub,
        run.start_qg_sub,
        run.s_sub_max,
    )

    @printf("%s stage1=%s/%s raw=%s hasv=%s mis=%.3e/%.3e vm=%.6f..%.6f\n",
        label,
        string(s1.term),
        string(s1.prim),
        s1.raw,
        string(s1.has_values),
        s1.max_p_mismatch,
        s1.max_q_mismatch,
        s1.vm_min,
        s1.vm_max,
    )

    @printf("%s stage2=%s/%s raw=%s hasv=%s mis=%.3e/%.3e vm=%.6f..%.6f sub=%.3e/%.3e\n",
        label,
        string(s2.term),
        string(s2.prim),
        s2.raw,
        string(s2.has_values),
        s2.max_p_mismatch,
        s2.max_q_mismatch,
        s2.vm_min,
        s2.vm_max,
        s2.pg_sub,
        s2.qg_sub,
    )
end

two_stage_trust_runs = Dict{String,Any}()
for (name, path) in [
    ("IEEE13", joinpath(@__DIR__, "..", "examples", "grids", "13_bus", "IEEE13Nodeckt.dss")),
    ("IEEE37", joinpath(@__DIR__, "..", "examples", "grids", "37_bus", "ieee37.dss")),
]
    with_gen = solve_power_balance_two_stage_trust(
        path;
        demand_mode = :pf_exact,
        remove_generators = false,
        stage1_trust_radius = 5e-4,
        stage2_trust_radius = 5e-2,
        stage2_objective_mode = :economic,
    )
    no_gen = solve_power_balance_two_stage_trust(
        path;
        demand_mode = :pf_exact,
        remove_generators = true,
        stage1_trust_radius = 5e-4,
        stage2_trust_radius = 5e-2,
        stage2_objective_mode = :feasibility,
    )

    two_stage_trust_runs[name] = (with_gen = with_gen, no_gen = no_gen)

    print_two_stage_trust_result("$name trust with_gen", with_gen)
    print_two_stage_trust_result("$name trust no_gen", no_gen)
end

IEEE13 trust with_gen start src=exact_pf start_mis=2.209e-09/6.016e-10 start_sub=6.528e-01/3.190e-01 lim=4.254e+00
IEEE13 trust with_gen stage1=LOCALLY_INFEASIBLE/INFEASIBLE_POINT raw=Infeasible_Problem_Detected hasv=true mis=1.205e-07/4.317e-08 vm=0.913698..0.999757
IEEE13 trust with_gen stage2=LOCALLY_INFEASIBLE/INFEASIBLE_POINT raw=Infeasible_Problem_Detected hasv=true mis=5.392e-05/5.636e-05 vm=0.900007..0.975395 sub=4.710e-01/3.030e-01
IEEE13 trust no_gen start src=exact_pf start_mis=2.209e-09/6.016e-10 start_sub=6.528e-01/3.190e-01 lim=4.254e+00
IEEE13 trust no_gen stage1=LOCALLY_INFEASIBLE/INFEASIBLE_POINT raw=Infeasible_Problem_Detected hasv=true mis=5.724e-08/1.549e-07 vm=0.913467..0.999896
IEEE13 trust no_gen stage2=LOCALLY_INFEASIBLE/INFEASIBLE_POINT raw=Infeasible_Problem_Detected hasv=true mis=7.220e-09/4.683e-09 vm=0.903734..0.991787 sub=6.519e-01/3.220e-01
IEEE37 trust with_gen start src=exact_pf start_mis=1.047e-08/2.860e-10 start_sub=8.672e-01/5.018e-01 lim=5.670e+00
I

In [90]:
for (name, path) in [
    ("IEEE13", joinpath(@__DIR__, "..", "examples", "grids", "13_bus", "IEEE13Nodeckt.dss")),
    ("IEEE37", joinpath(@__DIR__, "..", "examples", "grids", "37_bus", "ieee37.dss")),
]
    for lm in (false, true)
        for r2 in (5e-2, 2e-1, Inf)
            out = solve_power_balance_two_stage_trust(
                path;
                demand_mode = :pf_exact,
                remove_generators = true,
                use_limited_memory = lm,
                stage1_trust_radius = 5e-4,
                stage2_trust_radius = r2,
                stage2_objective_mode = :feasibility,
            )
            s2 = out.stage2
            @printf("%s no_gen lm=%s r2=%s => s2=%s/%s raw=%s mis=%.3e/%.3e vm=%.6f..%.6f\n",
                name,
                string(lm),
                isfinite(r2) ? @sprintf("%.2e", r2) : "Inf",
                string(s2.term),
                string(s2.prim),
                s2.raw,
                s2.max_p_mismatch,
                s2.max_q_mismatch,
                s2.vm_min,
                s2.vm_max,
            )
        end
    end
end

IEEE13 no_gen lm=false r2=5.00e-02 => s2=LOCALLY_INFEASIBLE/INFEASIBLE_POINT raw=Infeasible_Problem_Detected mis=1.844e-09/1.663e-09 vm=0.904455..0.992068
IEEE13 no_gen lm=false r2=2.00e-01 => s2=LOCALLY_INFEASIBLE/INFEASIBLE_POINT raw=Infeasible_Problem_Detected mis=2.313e-09/3.627e-09 vm=0.901524..0.989589
IEEE13 no_gen lm=false r2=Inf => s2=LOCALLY_INFEASIBLE/INFEASIBLE_POINT raw=Infeasible_Problem_Detected mis=8.542e-01/6.078e-01 vm=0.000000..1.100000
IEEE13 no_gen lm=true r2=5.00e-02 => s2=LOCALLY_INFEASIBLE/INFEASIBLE_POINT raw=Infeasible_Problem_Detected mis=7.220e-09/4.683e-09 vm=0.903734..0.991787
IEEE13 no_gen lm=true r2=2.00e-01 => s2=LOCALLY_INFEASIBLE/INFEASIBLE_POINT raw=Infeasible_Problem_Detected mis=1.366e-07/1.379e-07 vm=0.901835..0.991413
IEEE13 no_gen lm=true r2=Inf => s2=ITERATION_LIMIT/INFEASIBLE_POINT raw=Maximum_Iterations_Exceeded mis=1.501e+00/7.845e-01 vm=0.000831..1.121439
IEEE37 no_gen lm=false r2=5.00e-02 => s2=LOCALLY_INFEASIBLE/INFEASIBLE_POINT raw=Infea

In [92]:
for r1 in (1e-5, 5e-5, 1e-4, 5e-4, 1e-3, 1e-2)
    s1 = solve_power_balance_with_trust(
        joinpath(@__DIR__, "..", "examples", "grids", "13_bus", "IEEE13Nodeckt.dss");
        demand_mode = :pf_exact,
        remove_generators = true,
        objective_mode = :feasibility,
        use_limited_memory = false,
        trust_radius = r1,
    )
    @printf("IEEE13 no_gen stage1 r1=%s => %s/%s raw=%s mis=%.3e/%.3e vm=%.6f..%.6f\n",
        @sprintf("%.1e", r1),
        string(s1.term),
        string(s1.prim),
        s1.raw,
        s1.max_p_mismatch,
        s1.max_q_mismatch,
        s1.vm_min,
        s1.vm_max,
    )
end

IEEE13 no_gen stage1 r1=1.0e-05 => LOCALLY_INFEASIBLE/INFEASIBLE_POINT raw=Infeasible_Problem_Detected mis=4.098e-08/1.997e-08 vm=0.913680..1.000071
IEEE13 no_gen stage1 r1=5.0e-05 => LOCALLY_INFEASIBLE/INFEASIBLE_POINT raw=Infeasible_Problem_Detected mis=1.363e-07/2.963e-08 vm=0.913664..1.000057
IEEE13 no_gen stage1 r1=1.0e-04 => LOCALLY_INFEASIBLE/INFEASIBLE_POINT raw=Infeasible_Problem_Detected mis=7.637e-09/3.707e-09 vm=0.913644..1.000038
IEEE13 no_gen stage1 r1=5.0e-04 => LOCALLY_INFEASIBLE/INFEASIBLE_POINT raw=Infeasible_Problem_Detected mis=2.417e-08/5.734e-09 vm=0.913473..0.999900
IEEE13 no_gen stage1 r1=1.0e-03 => LOCALLY_INFEASIBLE/INFEASIBLE_POINT raw=Infeasible_Problem_Detected mis=6.194e-09/2.870e-09 vm=0.913254..0.999733
IEEE13 no_gen stage1 r1=1.0e-02 => LOCALLY_INFEASIBLE/INFEASIBLE_POINT raw=Infeasible_Problem_Detected mis=9.138e-09/7.517e-09 vm=0.909507..0.996460


In [91]:
for (name, pb) in sort(collect(power_balance_runs); by = x -> x[1])
    @assert pb.pf_converged

    @assert isfinite(pb.exact_max_p_mismatch)
    @assert isfinite(pb.exact_max_q_mismatch)
    @assert isfinite(pb.alloc_max_p_mismatch)
    @assert isfinite(pb.alloc_max_q_mismatch)

    @assert pb.exact_max_p_mismatch <= 1e-5
    @assert pb.exact_max_q_mismatch <= 1e-5
end

acceptable_term = Set([
    "OPTIMAL",
    "LOCALLY_SOLVED",
    "ALMOST_LOCALLY_SOLVED",
    "ITERATION_LIMIT",
    "NORM_LIMIT",
    "LOCALLY_INFEASIBLE",
    "INFEASIBLE",
    "INVALID_MODEL",
    "NUMERICAL_ERROR",
    "OTHER_ERROR",
])

detected_failures = String[]
for (name, bundle) in sort(collect(opf_runs); by = x -> x[1])
    m = bundle.metrics
    @assert string(m.term) in acceptable_term

    if m.has_values
        @assert isfinite(m.objective)
        @assert isfinite(m.vm_min)
        @assert isfinite(m.vm_max)
        @assert isfinite(m.max_p_mismatch)
        @assert isfinite(m.max_q_mismatch)
        @assert isfinite(m.max_angle_usage)
        @assert isfinite(m.max_current_usage)
        @assert isfinite(m.sub_s_usage)
    end

    if string(m.term) == "INVALID_MODEL" || string(m.term) == "INFEASIBLE" || string(m.term) == "LOCALLY_INFEASIBLE"
        push!(detected_failures, name)
    end
end

for (name, d) in sort(collect(deep_diagnostics); by = x -> x[1])
    q = d.line_quality
    @assert q.usable >= 0
    @assert q.skipped_singular >= 0
    @assert q.skipped_nonfinite_z >= 0
    @assert q.skipped_nonfinite_y >= 0
    @assert q.skipped_no_imax >= 0

    @assert isfinite(d.pb_alloc.max_p_mismatch) || !d.pb_alloc.has_values
    @assert isfinite(d.pb_alloc.max_q_mismatch) || !d.pb_alloc.has_values
    @assert isfinite(d.pb_exact.max_p_mismatch) || !d.pb_exact.has_values
    @assert isfinite(d.pb_exact.max_q_mismatch) || !d.pb_exact.has_values
end

println("Targeted power-balance diagnostics completed and no-slack OPF runs executed for IEEE13/37/123.")
if isempty(detected_failures)
    println("No hard feasibility/model failures detected in no-slack runs.")
else
    println("Feeder(s) with hard no-slack feasibility/model issues: " * join(detected_failures, ", "))
end

for (name, d) in sort(collect(deep_diagnostics); by = x -> x[1])
    pb_alloc_bad = d.pb_alloc.has_values && (d.pb_alloc.max_p_mismatch > 1e-3 || d.pb_alloc.max_q_mismatch > 1e-3)
    pb_exact_good = d.pb_exact.has_values && (d.pb_exact.max_p_mismatch <= 1e-5 && d.pb_exact.max_q_mismatch <= 1e-5)
    if pb_alloc_bad && pb_exact_good
        println("[" * name * "] evidence: demand mapping mismatch is a dominant feasibility driver.")
    end
    if d.line_quality.skipped_nonfinite_y > 0 || d.line_quality.skipped_nonfinite_z > 0
        println("[" * name * "] evidence: non-finite line parameter/admittance entries can invalidate branch constraints.")
    end
end

Targeted power-balance diagnostics completed and no-slack OPF runs executed for IEEE13/37/123.
Feeder(s) with hard no-slack feasibility/model issues: IEEE13, IEEE37
